# Setup

In [ ]:
import Pkg
using DiffOpt


using JuMP
using DiffOpt
using Gurobi
using ParametricOptInterface
using MathOptInterface
const MOI = MathOptInterface
const POI = ParametricOptInterface

using LinearAlgebra
using JLD2
using CSV, DataFrames
using Random
using Printf


using PowerModels
using Ipopt
using JuMP
using PowerModels
using JuMP
using Ipopt
using CSV, DataFrames
using LinearAlgebra, Random, SparseArrays, Dates
using MathOptInterface
const MOI = MathOptInterface


using CSV
using DataFrames
using JuMP
using Gurobi
using Ipopt
using LinearAlgebra
using Random
using SparseArrays
using XLSX
using SCS
using Distributions
using HTTP
using Dates
using Pkg
using Gurobi
using PowerModels
using Random
using Plots

# Setup system data

In [ ]:
TotalDays = 731
T = 24
K = 4
all_nodes = 1:24

Random.seed!(1234)

file_loc = "data"
case_file = "data/ieee24bus-testcase"

function add_day_index!(df, T)
    df[!, "Day Index"] = ceil.(Int, (1:nrow(df)) ./ T)
end

function add_day_hour!(df, T)
    n = nrow(df)
    df[!, "Day Index"] = ceil.(Int, (1:n) ./ T)
    df[!, "Hour"] = [(i - 1) % T + 1 for i in 1:n]
end


load_timeseries1 = CSV.read("$file_loc/load_timeseries1v2.csv", DataFrame)
add_day_index!(load_timeseries1, T)


data = PowerModels.parse_file(case_file)
PowerModels.standardize_cost_terms!(data, order=2)
PowerModels.calc_thermal_limits!(data)
ref = PowerModels.build_ref(data)[:it][:pm][:nw][0]

target_load_nodes = [14, 18, 7]

for bus in keys(ref[:bus_loads])
    if !(bus in target_load_nodes)
        ref[:bus_loads][bus] = []
    end
end


net_load_fixed = Dict{Int, Dict{Int, Dict{Int, Float64}}}()

for D in 1:TotalDays
    net_load_fixed[D] = Dict()
    for t in 1:T
        net_load_fixed[D][t] = Dict()
        for (i, bus) in ref[:bus]
            bus_loads = [
                ref[:load][l]["pd"] / 28.5 *
                load_timeseries1[load_timeseries1[!, "Day Index"] .== D, "real"] ./ 100
                for l in ref[:bus_loads][i] if !isempty(ref[:load][l])
            ]

            net_load_fixed[D][t][i] = isempty(bus_loads) ? 0.0 : bus_loads[1][t]
        end
    end
end


data = PowerModels.parse_file(case_file)
PowerModels.standardize_cost_terms!(data, order=2)
PowerModels.calc_thermal_limits!(data)
ref = PowerModels.build_ref(data)[:it][:pm][:nw][0]

for (l, branch) in data["branch"]
    if haskey(branch, "rate_a")
        branch["rate_a"] *= 0.9
    end
end

ref = PowerModels.build_ref(data)[:it][:pm][:nw][0]

target_load_nodes = [3, 5, 9, 16, 19, 20]

for bus in keys(ref[:bus_loads])
    if !(bus in target_load_nodes)
        ref[:bus_loads][bus] = []
    end
end


for (gen_id, gen_data) in ref[:gen]
    if gen_data["cost"][2] < 4
        original_cost = gen_data["cost"][2]
        gen_data["cost"][2] = rand(400:10:4000)
    end
end


load_timeseries = CSV.read("$file_loc/load_timeseriesv2.csv", DataFrame)
add_day_index!(load_timeseries, T)

wind_timeseries = CSV.read("$file_loc/wind_timeseries_scenario11v2.csv", DataFrame)
wind_timeseries2 = CSV.read("$file_loc/wind_timeseries_scenario21v2.csv", DataFrame)
wind_timeseries3 = CSV.read("$file_loc/wind_timeseries_scenario31v2.csv", DataFrame)
wind_timeseries4 = CSV.read("$file_loc/wind_timeseries_scenario41v2.csv", DataFrame)

wind_timeseries_real1 = CSV.read("$file_loc/wind_timeseries_real_year1v2.csv", DataFrame)
wind_timeseries_real2 = CSV.read("$file_loc/wind_timeseries_real_year2v2.csv", DataFrame)
wind_timeseries_real3 = CSV.read("$file_loc/wind_timeseries_real_year3v2.csv", DataFrame)
wind_timeseries_real4 = CSV.read("$file_loc/wind_timeseries_real_year4v2.csv", DataFrame)
wind_timeseries_real5 = CSV.read("$file_loc/wind_timeseries_real_year5v2.csv", DataFrame)

for df in [
    wind_timeseries,
    wind_timeseries2,
    wind_timeseries3,
    wind_timeseries4,
    wind_timeseries_real1,
    wind_timeseries_real2,
    wind_timeseries_real3,
    wind_timeseries_real4,
    wind_timeseries_real5
]
    add_day_hour!(df, T)
end

TotalDays = ceil(Int, nrow(wind_timeseries_real1) / T)


wind_farm_to_node_real = Dict(
    "windreal1" => 3,
    "windreal2" => 5,
    "windreal3" => 9,
    "windreal4" => 16,
    "windreal5" => 19,
    "windreal6" => 20,
    "windreal7" => 16
)

pwind_real = Dict{Int, Dict{Int, Dict{Int, Dict{Int, Float64}}}}()

for node in all_nodes
    pwind_real[node] = Dict()
    for t in 1:T
        pwind_real[node][t] = Dict()
        for D in 1:TotalDays
            pwind_real[node][t][D] = Dict()
            for k in 1:4
                pwind_real[node][t][D][k] = 0.0
            end
        end
    end
end

for (farm, node) in wind_farm_to_node_real
    for D in 1:TotalDays
        for t in 1:T
            filtered_rows = wind_timeseries_real1[(wind_timeseries_real1[!, "Day Index"] .== D) .&& (wind_timeseries_real1[!, "Hour"] .== t), :]
            power_value_real1 = isempty(filtered_rows) ? 0.0 : (filtered_rows[1, farm] / 15000) * 400

            filtered_rows = wind_timeseries_real2[(wind_timeseries_real2[!, "Day Index"] .== D) .&& (wind_timeseries_real2[!, "Hour"] .== t), :]
            power_value_real2 = isempty(filtered_rows) ? 0.0 : (filtered_rows[1, farm] / 15000) * 400

            filtered_rows = wind_timeseries_real3[(wind_timeseries_real3[!, "Day Index"] .== D) .&& (wind_timeseries_real3[!, "Hour"] .== t), :]
            power_value_real3 = isempty(filtered_rows) ? 0.0 : (filtered_rows[1, farm] / 15000) * 400

            filtered_rows = wind_timeseries_real4[(wind_timeseries_real4[!, "Day Index"] .== D) .&& (wind_timeseries_real4[!, "Hour"] .== t), :]
            power_value_real4 = isempty(filtered_rows) ? 0.0 : (filtered_rows[1, farm] / 15000) * 400

            pwind_real[node][t][D][1] = power_value_real1
            pwind_real[node][t][D][2] = power_value_real2
            pwind_real[node][t][D][3] = power_value_real3
            pwind_real[node][t][D][4] = power_value_real4
        end
    end
end


wind_farm_to_node = Dict(
    "Wind Farm 1 (MW)" => 3,
    "Wind Farm 2 (MW)" => 5,
    "Wind Farm 3 (MW)" => 9,
    "Wind Farm 4 (MW)" => 16,
    "Wind Farm 5 (MW)" => 19,
    "Wind Farm 6 (MW)" => 20,
    "Offshore Wind (MW)" => 17
)

TotalDays = ceil(Int, nrow(wind_timeseries) / T)

pwind = Dict{Int, Dict{Int, Dict{Int, Dict{Int, Float64}}}}()

for node in all_nodes
    pwind[node] = Dict()
    for t in 1:T
        pwind[node][t] = Dict()
        for D in 1:TotalDays
            pwind[node][t][D] = Dict()
            for k in 1:4
                pwind[node][t][D][k] = 0.0
            end
        end
    end
end

for (farm, node) in wind_farm_to_node
    for D in 1:TotalDays
        for t in 1:T
            filtered_rows1 = wind_timeseries[(wind_timeseries[!, "Day Index"] .== D) .&& (wind_timeseries[!, "Hour"] .== t), :]
            power_value1 = isempty(filtered_rows1) ? 0.0 : (filtered_rows1[1, farm] / 15000) * 400

            filtered_rows2 = wind_timeseries2[(wind_timeseries2[!, "Day Index"] .== D) .&& (wind_timeseries2[!, "Hour"] .== t), :]
            power_value2 = isempty(filtered_rows2) ? 0.0 : (filtered_rows2[1, farm] / 15000) * 400

            filtered_rows3 = wind_timeseries3[(wind_timeseries3[!, "Day Index"] .== D) .&& (wind_timeseries3[!, "Hour"] .== t), :]
            power_value3 = isempty(filtered_rows3) ? 0.0 : (filtered_rows3[1, farm] / 15000) * 400

            filtered_rows4 = wind_timeseries4[(wind_timeseries4[!, "Day Index"] .== D) .&& (wind_timeseries4[!, "Hour"] .== t), :]
            power_value4 = isempty(filtered_rows4) ? 0.0 : (filtered_rows4[1, farm] / 15000) * 400

            pwind[node][t][D][1] = power_value1
            pwind[node][t][D][2] = power_value2
            pwind[node][t][D][3] = power_value3
            pwind[node][t][D][4] = power_value4
        end
    end
end


loadd = Dict{Int, Dict{Int, Dict{Int, Dict{Int, Float64}}}}()

for D in 1:TotalDays
    loadd[D] = Dict()
    for t in 1:T
        for (i, bus) in ref[:bus]
            loadd[D][t] = get(loadd[D], t, Dict())
            loadd[D][t][i] = get(loadd[D][t], i, Dict())

            for k in 1:K
                column_name = Symbol("load_forecast_k$k")

                activeload = if !isempty(ref[:bus_loads][i])
                    [
                        ref[:load][l]["pd"] / 28.5 *
                        load_timeseries[load_timeseries[!, "Day Index"] .== D, column_name]
                        for l in ref[:bus_loads][i]
                    ][1][t]
                else
                    0.0
                end

                loadd[D][t][i][k] = activeload
            end
        end
    end
end


net_load = Dict{Int, Dict{Int, Dict{Int, Dict{Int, Float64}}}}()

for D in 1:TotalDays
    net_load[D] = Dict()
    for t in 1:T
        for (i, bus) in ref[:bus]
            net_load[D][t] = get(net_load[D], t, Dict())
            net_load[D][t][i] = get(net_load[D][t], i, Dict())

            for k in 1:K
                net_load[D][t][i][k] = (loadd[D][t][i][k] - pwind[i][t][D][k]) / 100
            end
        end
    end
end


bus_ids = sort(collect(keys(ref[:bus])))
numBuses = length(bus_ids)
bus_index = Dict(bid => k for (k, bid) in enumerate(bus_ids))

maxK = 4

realload = zeros(TotalDays, T, numBuses, maxK)

for D in 1:TotalDays
    real_vec = Vector(load_timeseries1[load_timeseries1[!, "Day Index"] .== D, "real"])
    @assert length(real_vec) == T "load_timeseries1 day $D must have T rows"

    for t in 1:T
        for bid in bus_ids
            load_series = [
                (ref[:load][l]["pd"] / 28.5) .* real_vec
                for l in ref[:bus_loads][bid] if !isempty(ref[:load][l])
            ]

            base_load_t = isempty(load_series) ? 0.0 : sum(s[t] for s in load_series)

            nK = haskey(pwind_real, bid) &&
                 haskey(pwind_real[bid], t) &&
                 haskey(pwind_real[bid][t], D) ? length(pwind_real[bid][t][D]) : 0

            for k in 1:maxK
                wind_t = k <= nK ? pwind_real[bid][t][D][k] : 0.0
                realload[D, t, bus_index[bid], k] = (base_load_t - wind_t) / 100.0
            end
        end
    end
end

# Capacity data

In [ ]:
scale1 = 0.0001


cand_wind_data = Dict(
    "Wind Gen 1" => (bus=3,  pmax=18, c_fix=5000000 * scale1, c_var=9000000 * scale1),
     "Wind Gen 2" => (bus=5, pmax=15, c_fix=4000000  * scale1, c_var=8000000 * scale1),
    "Wind Gen 3" => (bus=9, pmax=20, c_fix=3500000 * scale1, c_var=8500000 * scale1),
    "Wind Gen 4" => (bus=16,  pmax=15, c_fix=5000000 * scale1, c_var=9000000 * scale1),
    "Wind Gen 5" => (bus=19, pmax=20, c_fix=4000000 * scale1, c_var=10000000 * scale1),
     "Wind Gen 6" => (bus=20, pmax=14, c_fix=3500000 * scale1, c_var=8500000 * scale1)
)
RENW = collect(keys(cand_wind_data))
bus_of_wind = Dict(b => cand_wind_data[b].bus for b in RENW)



battery_data = Dict(
    "Battery 16" => (bus = 16, E = 6.4, p_ch = 1.600, p_dis = 1.60, c_var_b=105000 * scale1, c_var_E = 12000000 * scale1, c_opr_b = 150 * scale1),
    "Battery 20" => (bus = 20, E = 3.8, p_ch = 0.950, p_dis = 0.95, c_var_b=110000 * scale1,c_var_E = 10000000 * scale1, c_opr_b=250 * scale1 ),
    "Battery 7"  => (bus = 7,  E =  1.6, p_ch = 0.4000, p_dis = 0.40, c_var_b=100000 * scale1 ,c_var_E = 11000000 * scale1, c_opr_b = 400* scale1),
    "Battery 24" => (bus = 24, E = 2, p_ch = 0.50, p_dis = 0.50, c_var_b=80000 * scale1, c_var_E = 17000000 * scale1, c_opr_b= 500 * scale1),
)
BATS = collect(keys(battery_data))
bus_of_bat = Dict(b => battery_data[b].bus for b in BATS)




fuel_gen_data = Dict(
    "Fuel Gen 1" => (bus=8,  pmax=3.500,  c_fix=38000.0 * scale1, c_var=4000000* scale1, c_can_fuel=2000.0 * scale1, ru=0.8 , rd=0.8),
    "Fuel Gen 2" => (bus=21,  pmax=2.800,  c_fix=39000.0 * scale1, c_var=4200000 * scale1, c_can_fuel=1200.0 * scale1, ru=0.8 , rd=0.8),
    "Fuel Gen 3" => (bus=9,  pmax=3.1000, c_fix=14000.0 * scale1, c_var=2500000 * scale1, c_can_fuel=2000.0 * scale1, ru=0.8, rd=0.8),
   "Fuel Gen 4" => (bus=17, pmax=4.1200, c_fix=20000.0 * scale1, c_var=2500000 * scale1, c_can_fuel=1200.0 * scale1, ru=0.8 , rd=0.8),
    "Fuel Gen 5" => (bus=19, pmax=3.1500, c_fix=15500.0 * scale1, c_var=2500000 * scale1, c_can_fuel=1700.0 * scale1, ru=0.8 , rd=0.8 ),
    "Fuel Gen 6" => (bus=20, pmax=2.2000, c_fix=18000.0 * scale1, c_var=3400000 * scale1, c_can_fuel=2100.0 * scale1, ru=0.8 , rd=0.8),
    "Fuel Gen 7" => (bus=16, pmax=4.750,  c_fix=36500.0 * scale1, c_var=3500000 * scale1, c_can_fuel=1200.0 * scale1, ru=0.8 , rd=0.8)
)

_get(g, sym::Symbol, default=nothing) =
    hasproperty(g, sym) ? getfield(g, sym) :
    g isa AbstractDict   ? get(g, String(sym), default) :
    default


_get(g, sym::Symbol, default=nothing) =
    hasproperty(g, sym) ? getfield(g, sym) :
    g isa AbstractDict   ? get(g, String(sym), default) :
    default

function merge_candidates_into_ref!(ref::Dict{Symbol,Any},
                                    fuel_gen_data::AbstractDict{<:AbstractString,<:Any})
    name_to_id = Dict{String,Int}()
    existing_ids = collect(keys(ref[:gen]))
    next_id = isempty(existing_ids) ? 1 : maximum(existing_ids) + 1

    for name in keys(fuel_gen_data)
        g = fuel_gen_data[name]
        bus  = _get(g, :bus)
        pmax = _get(g, :pmax)
        cvar = _get(g, :c_var)
        cfuel = _get(g, :c_can_fuel)
        isnothing(bus)  && error("Generator '$name' missing field :bus")
        isnothing(pmax) && error("Generator '$name' missing field :pmax")
        isnothing(cvar) && error("Generator '$name' missing field :c_var")
        isnothing(cfuel) && error("Generator '$name' missing field :c_can_fuel")
        pmin = something(_get(g, :pmin, nothing), 0.0)

        gid = next_id; next_id += 1
        ref[:gen][gid] = Dict(
            "gen_bus"   => bus,
            "pmin"      => pmin,
            "pmax"      => pmax,
            "status"    => 1,
            "name"      => name,
            "cost"      => [0.0, cfuel],
            "candidate" => 1
        )
        push!(get!(ref[:bus_gens], bus, Int[]), gid)
        name_to_id[name] = gid
    end
    return name_to_id
end

gen_id_map = merge_candidates_into_ref!(ref, fuel_gen_data)
NEWGENS = collect(keys(fuel_gen_data))

# cost maps
c_g_fix_id = Dict(id => fuel_gen_data[name].c_fix for (name, id) in gen_id_map)
c_g_var_id = Dict(id => fuel_gen_data[name].c_var for (name, id) in gen_id_map)
CAND = sort!(collect(keys(c_g_fix_id)))
@assert CAND == sort(collect(keys(c_g_var_id)))

# sets
ALLGENS = sort!(collect(keys(ref[:gen])))
EXIST   = setdiff(ALLGENS, CAND)

# candidate lines
CANDLINES = Dict(
    "CL_1" => (f_bus = 3,  t_bus = 9,  x_pu = 0.12, rate_MW = 2.000 * 0.9, c_fix = 20000.0 * scale1, c_var = 10000 * scale1),
    "CL_2" => (f_bus = 16, t_bus = 19, x_pu = 0.10, rate_MW = 2.500 * 0.9, c_fix = 22000.0 * scale1, c_var = 9000.5 * scale1)
)
CL   = collect(keys(CANDLINES))
fbus = Dict(l => CANDLINES[l].f_bus for l in CL)
tbus = Dict(l => CANDLINES[l].t_bus for l in CL)
Bx   = Dict(l => 1.0 / CANDLINES[l].x_pu for l in CL)
Fmax = Dict(l => CANDLINES[l].rate_MW for l in CL)
c_l_fix = Dict(l => CANDLINES[l].c_fix for l in CL)
c_l_var = Dict(l => CANDLINES[l].c_var for l in CL)

function merge_candidate_lines_into_ref!(ref, CANDLINES)
    name_to_id = Dict{String,Int}()
    existing = collect(keys(ref[:branch]))
    next_id = isempty(existing) ? 1 : maximum(existing) + 1
    for name in keys(CANDLINES)
        cl = CANDLINES[name]
        f  = cl.f_bus; t = cl.t_bus; x = cl.x_pu; F = cl.rate_MW
        l_id = next_id; next_id += 1
        ref[:branch][l_id] = Dict(
            "name"      => name,
            "f_bus"     => f,
            "t_bus"     => t,
            "br_r"      => 0.0,
            "br_x"      => x,
            "br_b"      => 0.0,
            "tap"       => 1.0,
            "shift"     => 0.0,
            "rate_a"    => F,
            "status"    => 1,
            "candidate" => 1,
            "c_fix"     => cl.c_fix,
            "c_var"     => cl.c_var
        )
        push!(ref[:arcs_from], (l_id, f, t))
        push!(get!(ref[:bus_arcs], f, Vector{Tuple{Int,Int,Int}}()), (l_id, f, t))
        push!(get!(ref[:bus_arcs], t, Vector{Tuple{Int,Int,Int}}()), (l_id, t, f))
        name_to_id[name] = l_id
    end
    return name_to_id
end

line_id_map = merge_candidate_lines_into_ref!(ref, CANDLINES)
c_l_fix_id = Dict(id => CANDLINES[name].c_fix for (name, id) in line_id_map)
c_l_var_id = Dict(id => CANDLINES[name].c_var for (name, id) in line_id_map)
CANDL = sort!(collect(keys(c_l_fix_id)))
@assert CANDL == sort(collect(keys(c_l_var_id)))

ref[:arcs_from] = Tuple{Int,Int,Int}[]
ref[:bus_arcs]  = Dict{Int, Vector{Tuple{Int,Int,Int}}}()
for (l, br) in ref[:branch]
    f = br["f_bus"]; to = br["t_bus"]
    push!(ref[:arcs_from], (l, f, to))
    push!(get!(ref[:bus_arcs], f, Vector{Tuple{Int,Int,Int}}()), (l, f, to))
    push!(get!(ref[:bus_arcs], to, Vector{Tuple{Int,Int,Int}}()), (l, to, f))
end

ALLLINES = sort!(collect(keys(ref[:branch])))
EXISTL   = setdiff(ALLLINES, CANDL)

B_line = Dict{Int,Float64}()
for (l, br) in ref[:branch]
    g, b = PowerModels.calc_branch_y(br)
    B_line[l] = b
end


Gens       = collect(ALLGENS)
CandGens   = collect(CAND)
ExistGens  = collect(EXIST)
Lines      = collect(ALLLINES)
CandLines  = collect(CANDL)
ExistLines = collect(EXISTL)


# Parameters

In [ ]:
scale1 = 0.0001
M = 100000
c_cur_prime=2000 * scale1
c_shed_prime = 1000000 * scale1
c_cur=2000 * scale1
c_shed = 1000000 * scale1
TotalDays = 365
r = 0.25
TotalYears = 1
T=24
DD = 1:2:TotalDays


eta = [0.5, 0.20, 0.20, 0.20] 
loadyearly = [1, 1.15, 2.1, 3.15]

for g in ExistGens
    if !haskey(ref[:gen][g], "ramp_up")
        ref[:gen][g]["ramp_up"] = 0.90 * ref[:gen][g]["pmax"]
    end
    if !haskey(ref[:gen][g], "ramp_down")
        ref[:gen][g]["ramp_down"] = 0.90 * ref[:gen][g]["pmax"]
    end
end


RU = Dict(g => ref[:gen][g]["ramp_up"]   for g in Gens)
RD = Dict(g => ref[:gen][g]["ramp_down"] for g in Gens)




model = Model(Gurobi.Optimizer)

for g in ExistGens
    if !haskey(ref[:gen][g], "ramp_up")
        ref[:gen][g]["ramp_up"] = 0.90 * ref[:gen][g]["pmax"]
    end
    if !haskey(ref[:gen][g], "ramp_down")
        ref[:gen][g]["ramp_down"] = 0.90 * ref[:gen][g]["pmax"]
    end
end


RU = Dict(g => ref[:gen][g]["ramp_up"]   for g in Gens)
RD = Dict(g => ref[:gen][g]["ramp_down"] for g in Gens)



DF = Dict(y => 1 / (1 + r)^(y) for y in 1:TotalYears)

# Build: gen_var_cost (energy $/MWh) for all gens
gen_var_cost = Dict(
    i => (i in CandGens ?
        get(ref[:gen][i], "cost", [0.0, 0.0])[2]  :
        get(ref[:gen][i], "cost", [0.0, 0.0])[2] * scale1 )
    for i in Gens
)


# Candidate fuel costs map (c_fuel or c_can_fuel)
c_fuel_id = Dict{Int,Float64}()
for (name, gid) in gen_id_map
    cval = get(fuel_gen_data[name], :c_fuel, get(fuel_gen_data[name], :c_can_fuel, nothing))
    isnothing(cval) && error("Candidate '$name' missing :c_fuel (or :c_can_fuel) value.")
    c_fuel_id[gid] = cval
end

# Reserve costs 
r_cost_up = Dict(i => 1.5 * gen_var_cost[i] for i in Gens)
r_cost_dn = Dict(i => 0.7 * gen_var_cost[i] for i in Gens)


# Building models + Gradients

In [ ]:
function flatten_vars(container)
    vars = VariableRef[]
    for v in container    
        push!(vars, v)
    end
    return vars
end



# STEP 0 — initial parameter values for θ and pg
d_sample = rand(DD)
days_subset = [d_sample]
YEARS = 1:TotalYears

# 1) Investment capacities for candidate generators
p_new_data = Dict{Tuple{Int,Int},Float64}()
for g in CandGens, y in YEARS
    p_new_data[(g,y)] = 0.0  
end

# 2) Investment capacities for candidate lines
f_cand_data = Dict{Tuple{Int,Int},Float64}()
for l in CandLines, y in YEARS
    f_cand_data[(l,y)] = 0.0  
end

# 3) Wind capacity per renewable site
p_wind_cap_data = Dict{Tuple{String,Int},Float64}()
for g in RENW, y in YEARS
    p_wind_cap_data[(g,y)] = 0.0   
end

# 4) Battery power capacity per site
P_BAT_cap_data = Dict{Tuple{String,Int},Float64}()
for b in BATS, y in YEARS
    P_BAT_cap_data[(b,y)] = 0.0    
end

# 5) Battery energy capacity per site
E_BAT_cap_data = Dict{Tuple{String,Int},Float64}()
for b in BATS, y in YEARS
    E_BAT_cap_data[(b,y)] = 0.0    
end

# 6) Coupling pg data (middle → inner)
pg_data = Dict{NTuple{4,Int},Float64}()
for i in Gens, t in 1:T, D in DD, y in YEARS
    pg_data[(i,t,D,y)] = 0.0
end


function numeric_jacobian_forward!(
    J::AbstractMatrix,
    model::Model,
    θ::Vector{VariableRef},     
    y::Vector{VariableRef};     
    ε::Float64 = 1e-6,
)

    # Base solution at current θ
    base_y = value.(y)

    # Read current parameter values using POI.ParameterValue
    base_θ = [
        MOI.get(model, POI.ParameterValue(), JuMP.index(p))
        for p in θ
    ]

    for j in eachindex(θ)
        idx = JuMP.index(θ[j])
        θ0  = base_θ[j]

        MOI.set(model, POI.ParameterValue(), idx, θ0 + ε)
        optimize!(model)

        # Forward diff for column j
        for i in eachindex(y)
            J[i, j] = (value(y[i]) - base_y[i]) / ε
        end

        # reset θ_j
        MOI.set(model, POI.ParameterValue(), idx, θ0)
    end

    # re-solve at base θ
    optimize!(model)

    return J
end


const GRB_ENV = Gurobi.Env()

function new_diffopt_model()
    return Model(() -> DiffOpt.diff_optimizer(
        () -> Gurobi.Optimizer(GRB_ENV)
    ))
end


function diffopt_vjp_parameters!(
    model::Model,
    θ::Vector{VariableRef},
    y::Vector{VariableRef},
    Δy::AbstractVector{<:Real},
)
    length(y) == length(Δy) || error("Seed Δy length must match y length.")
    optimize!(model)

    # 1) seed reverse on output primals y with Δy
    @inbounds for i in eachindex(y)
        DiffOpt.set_reverse_variable(model, y[i], Δy[i])
    end

    # 2) reverse differentiate
    DiffOpt.reverse_differentiate!(model)

    # 3) read sensitivities wrt parameters θ
    g = zeros(Float64, length(θ))
    @inbounds for i in eachindex(θ)
        g[i] = DiffOpt.get_reverse_parameter(model, θ[i])
    end
    return g
end


function diffopt_vjp_parameters!(
    model::Model,
    θ_all::Vector{VariableRef},
    y::Vector{VariableRef},
    Δy::AbstractVector{<:Real},
    ::Val{:all},
)
    length(y) == length(Δy) || error("Seed Δy length must match y length.")
    optimize!(model)

    @inbounds for i in eachindex(y)
        DiffOpt.set_reverse_variable(model, y[i], Δy[i])
    end

    DiffOpt.reverse_differentiate!(model)

    g = zeros(Float64, length(θ_all))
    @inbounds for i in eachindex(θ_all)
        g[i] = DiffOpt.get_reverse_parameter(model, θ_all[i])
    end
    return g
end


function add_middle_objective!(
    model,
    ref, CandGens, CandLines, RENW, BATS,
    TotalYears, T, DD,
    DF::Dict{Int,Float64},
    ALLGENS,
    gen_var_cost,
    c_shed_prime, c_cur_prime,
    battery_data, cand_wind_data,
    c_g_fix_id, c_g_var_id,
    c_l_fix_id, c_l_var_id,
    p_new, f_cand, p_wind_cap, P_BAT_cap, E_BAT_cap,
    pg, p_shed_da, p_cur_da, P_BAT_da,
    windpower_da,
    P_DIS_da,
        P_CH_da;    
    # penalty_ren::Float64;     
    days_subset::AbstractVector{Int} = collect(DD),
)
    YEARS   = 1:TotalYears
    obj_expr = 0.0
    scale = 1
    for y in YEARS
       
        # DA operating cost + renewable shortfall penalty
        op_y = 0.0
         
        # DA operating cost for sampled days
        for d in days_subset, t in 1:T
            # DA generation cost
            for g in ALLGENS
                op_y += (gen_var_cost[g]* scale) * pg[g,t,d,y]
            end

            # DA shedding / curtailment
            for i in keys(ref[:bus])
                op_y += c_shed_prime* scale * p_shed_da[i,t,d,y] +
                        c_cur_prime* scale * p_cur_da[i,t,d,y]
            end

            # DA battery operating cost
            for b in BATS
                op_y += battery_data[b].c_opr_b * scale  * (P_DIS_da[b,t,d,y]+P_CH_da[b,t,d,y])
            end
        end
        obj_expr += DF[y] * (op_y)
    end

    @objective(model, Min, obj_expr)
    return nothing
end



function add_inner_objective!(
    model, ref, BATS,
    TotalYears, T, days_subset, ALLGENS, DF,
    r_cost_up, r_cost_dn,
    c_shed_prime, c_cur_prime,
    battery_data,
    r_up, r_down,
    p_shed_rt, p_cur_rt,
    P_BAT_rt,
windpower_rt,
        P_DIS_rt,
        P_CH_rt
)
    YEARS    = 1:TotalYears
    obj_expr = 0.0
    scale    = 1.0

    for y in YEARS
        op_y = 0.0

        for d in days_subset, t in 1:T
            # Reserve costs
            for g in ALLGENS
                op_y += scale * (r_cost_up[g] * r_up[g,t,d,y] - r_cost_dn[g] * r_down[g,t,d,y])
            end

            # Shedding / curtailment
            for i in keys(ref[:bus])
                op_y += scale * (c_shed_prime * p_shed_rt[i,t,d,y] + c_cur_prime * p_cur_rt[i,t,d,y])
            end

            # Battery operating cost/benefit
            for b in BATS
                op_y += scale * battery_data[b].c_opr_b  * (P_DIS_rt[b,t,d,y]+P_CH_rt[b,t,d,y])
            end
        end
        obj_expr += DF[y] * (op_y)
    end

    @objective(model, Min, obj_expr)
    return nothing
end

#build_middle_model 
function build_middle_model(ref, CandGens, CandLines, RENW, BATS,
                            TotalYears, T, DD, DF, r, eta,
                            cand_wind_data, battery_data,
                            ref_data, c_g_fix_id, c_g_var_id,
                            c_l_fix_id, c_l_var_id,
                            gen_var_cost, r_cost_up, r_cost_dn,
                            loadyearly, loadd, net_load, net_load_fixed,
                            pwind, pwind_real, bus_of_wind, bus_of_bat,
                            ALLGENS, EXISTGENS, EXISTLINES, Gens,
                            B_line, CandLinesDict,
                            c_shed_prime, c_cur_prime;
                            p_new_data,
                            f_cand_data,
                            p_wind_cap_data,
                            P_BAT_cap_data,
                            E_BAT_cap_data,
                            scenario_index::Int = 1,
                            days_subset::AbstractVector{Int} = collect(DD),
)

    model = new_diffopt_model()
    YEARS = 1:TotalYears


    # PARAMETERS
    @variable(model, p_new[g in CandGens, y in YEARS] in
        MOI.Parameter(get(p_new_data,(g,y),0.0)))

    @variable(model, f_cand[l in CandLines, y in YEARS] in
        MOI.Parameter(get(f_cand_data,(l,y),0.0)))

    @variable(model, p_wind_cap[w in RENW, y in YEARS] in
        MOI.Parameter(get(p_wind_cap_data,(w,y),0.0)))

    @variable(model, P_BAT_cap[b in BATS, y in YEARS] in
        MOI.Parameter(get(P_BAT_cap_data,(b,y),0.0)))

    @variable(model, E_BAT_cap[b in BATS, y in YEARS] in
        MOI.Parameter(get(E_BAT_cap_data,(b,y),0.0)))


    # DA VARIABLES

    @variable(model, va[i in keys(ref[:bus]), t=1:T, d in days_subset, y in YEARS])
    @variable(model, pg[g in Gens, t=1:T, d in days_subset, y in YEARS] >= 0)

    @variable(model,
        -ref[:branch][l]["rate_a"] <=
        p_da[(l,i,j) in ref[:arcs_from], t=1:T, d in days_subset, y in YEARS] <=
        ref[:branch][l]["rate_a"]
    )

    @variable(model, p_shed_da[i in keys(ref[:bus]), t=1:T, d in days_subset, y in YEARS] >= 0)
    @variable(model, p_cur_da[i in keys(ref[:bus]),  t=1:T, d in days_subset, y in YEARS] >= 0)

    @variable(model, e_da[b in BATS, t=1:T, d in days_subset, y in YEARS] >= 0)
    @variable(model, P_BAT_da[b in BATS, t=1:T, d in days_subset, y in YEARS])
    @variable(model, windpower_da[w in RENW, t=1:T, d in days_subset, y in YEARS]>= 0)



    # EXISTING LINE FLOWS
    for l in EXISTLINES, d in days_subset, t=1:T, y in YEARS
        br   = ref[:branch][l]
        fbus = br["f_bus"]
        tbus = br["t_bus"]

        @constraint(model,
            p_da[(l,fbus,tbus),t,d,y] ==
            -B_line[l]*(va[fbus,t,d,y] - va[tbus,t,d,y])
        )
    end


    # SIGNED FLOW DICTIONARY (for nodal balance)
    p_expr = Dict(
        ((l,i,j),t,d,y) => +p_da[(l,i,j),t,d,y]
        for (l,i,j) in ref[:arcs_from], t=1:T, d in days_subset, y in YEARS
    )
    p_expr = merge(p_expr,
        Dict(
            ((l,j,i),t,d,y) => -p_da[(l,i,j),t,d,y]
            for (l,i,j) in ref[:arcs_from], t=1:T, d in days_subset, y in YEARS
        )
    )


    # CANDIDATE LINE FLOWS 
    for l in CandLines, d in days_subset, t=1:T, y in YEARS
        br   = ref[:branch][l]
        fbus = br["f_bus"]
        tbus = br["t_bus"]

        @constraint(model,
            p_da[(l,fbus,tbus),t,d,y] ==
            -B_line[l]*(va[fbus,t,d,y] - va[tbus,t,d,y])
        )

        @constraint(model,  p_da[(l,fbus,tbus),t,d,y] <=  f_cand[l,y])
        @constraint(model, -p_da[(l,fbus,tbus),t,d,y] <=  f_cand[l,y])
    end


    # GENERATION LIMITS
    for g in CandGens, d in days_subset, t=1:T, y in YEARS
        @constraint(model, pg[g,t,d,y] <= p_new[g,y])
    end

    for g in EXISTGENS, d in days_subset, t=1:T, y in YEARS
        @constraint(model, pg[g,t,d,y] <= ref[:gen][g]["pmax"])
    end

    #ramp

   for g in CandGens, d in days_subset, y in YEARS, t=2:T
       @constraint(model, pg[g,t,d,y] - pg[g,t-1,d,y] <= 0.8 * p_new[g,y])
       @constraint(model, pg[g,t-1,d,y] - pg[g,t,d,y] <= p_new[g,y])
   end


    for g in EXISTGENS, d in days_subset, t=2:T, y in YEARS
         @constraint(model, pg[g,t,d,y] - pg[g,t-1,d,y] <= ref[:gen][g]["pmax"]*0.8 )
         @constraint(model, pg[g,t,d,y] - pg[g,t+1,d,y] <= ref[:gen][g]["pmax"])
    end
    
    
    # SLACK BUS
    for (slack,_) in ref[:ref_buses], d in days_subset, t=1:T, y in YEARS
        @constraint(model, va[slack,t,d,y] == 0)
    end


    # BATTERY
    @variable(model, P_DIS_da[b in BATS, t in 1:T, d in days_subset, y in YEARS] >= 0)
    @variable(model, P_CH_da[b in BATS, t in 1:T, d in days_subset, y in YEARS] >= 0)
    for b in BATS, d in days_subset, t=1:T, y in YEARS
        @constraint(model, e_da[b,t,d,y] <= E_BAT_cap[b,y])
        @constraint(model, P_BAT_da[b,t,d,y] <=  P_BAT_cap[b,y])
        @constraint(model, P_BAT_da[b,t,d,y] >= -P_BAT_cap[b,y])
        @constraint(model, P_BAT_da[b,t,d,y] == P_DIS_da[b,t,d,y] - P_CH_da[b,t,d,y])
    end


    for b in BATS, d in days_subset, t=2:T, y in YEARS
        @constraint(model,
            e_da[b,t,d,y] == e_da[b,t-1,d,y] - P_BAT_da[b,t,d,y]
        )
    end

    @constraint(model,[b in BATS, d in days_subset, y in YEARS],
        e_da[b,1,d,y] == 0.5*E_BAT_cap[b,y])

    @constraint(model,[b in BATS, d in days_subset, y in YEARS],
        e_da[b,T,d,y] == 0.5*E_BAT_cap[b,y])



    for g in RENW,d in days_subset, t in 1:T, y in YEARS
       bus = bus_of_wind[g]
       @constraint(model,
           windpower_da[g,t,d,y] <= (pwind[bus][t][d][y] / 400) * p_wind_cap[g,y]
       )
       @constraint(model, 0 <= windpower_da[g,t,d,y])
    end
    
    # NODAL BALANCE 
    for y in YEARS, d in days_subset, t=1:T
        for (i,_) in ref[:bus]

            # wind injection 
            wind_term = sum(
                windpower_da[w,t,d,y]
                for w in RENW if bus_of_wind[w] == i;
                init = 0.0
            )

            @constraint(model,
                sum(p_expr[a,t,d,y] for a in ref[:bus_arcs][i]) ==

                sum(pg[g,t,d,y] for g in ref[:bus_gens][i]) +

                # battery withdrawal/injection
                sum(P_BAT_da[b,t,d,y] for b in BATS if bus_of_bat[b] == i;
                    init = 0.0) -
                 (loadyearly[y] * (loadd[d][t][i][y]/100) + loadyearly[y] * net_load_fixed[d][t][i]) +
                wind_term +

                p_shed_da[i,t,d,y] 
               
            )
        end
    end


########################################################################
    for (i,_) in ref[:bus], d in days_subset, t in 1:T, y in YEARS
        @constraint(model, p_shed_da[i,t,d,y] <= loadyearly[y] * (loadd[d][t][i][y]/100) + loadyearly[y] * net_load_fixed[d][t][i])
    end
   

for y in YEARS, d in days_subset, t in 1:T, (i,_) in ref[:bus]
    gens_at_bus = [g for g in RENW if bus_of_wind[g] == i]

    if isempty(gens_at_bus)
        @constraint(model, p_cur_da[i,t,d,y] == 0)
      else
        @constraint(model,
            p_cur_da[i,t,d,y] ==
            sum((pwind[i][t][d][y] / 400) * p_wind_cap[g,y] - windpower_da[g,t,d,y]
                for g in gens_at_bus)
        )
    end
end

    # OBJECTIVE 
    add_middle_objective!(
        model,
        ref, CandGens, CandLines, RENW, BATS,
        TotalYears, T, DD, DF,
        ALLGENS,
        gen_var_cost,
        c_shed_prime, c_cur_prime,
        battery_data, cand_wind_data,
        c_g_fix_id, c_g_var_id,
        c_l_fix_id, c_l_var_id,
        p_new, f_cand, p_wind_cap, P_BAT_cap, E_BAT_cap,
        pg, p_shed_da, p_cur_da, P_BAT_da,
        windpower_da,
        P_DIS_da,
        P_CH_da;
        days_subset = days_subset,
    )

    return model
end






# build_inner_model (FINAL CORRECTED)
function build_inner_model(ref, CandGens, CandLines, RENW, BATS,
                           TotalYears, T, DD,
                           r_cost_up, r_cost_dn,
                           battery_data, ref_data,
                           cand_wind_data,               
                           loadyearly, loadd, realload, net_load_fixed, pwind_real,
                           bus_of_wind, bus_of_bat,
                           ALLGENS, Gens, EXISTGENS,
                           EXISTLINES, CandLinesDict,
                           B_line, c_shed_prime, c_cur_prime,
                           eta;                           
                           p_new_data,
                           f_cand_data,
                           p_wind_cap_data,
                           P_BAT_cap_data,
                           E_BAT_cap_data,
                           pg_data,
                           days_subset = collect(DD)
)

    model = new_diffopt_model()
    YEARS = 1:TotalYears

   
    # PARAMETERS
    @variable(model, p_new[g in CandGens, y in YEARS] in
        MOI.Parameter(get(p_new_data, (g,y), 0.0)))

    @variable(model, f_cand[l in CandLines, y in YEARS] in
        MOI.Parameter(get(f_cand_data, (l,y), 0.0)))

    @variable(model, p_wind_cap[w in RENW, y in YEARS] in
        MOI.Parameter(get(p_wind_cap_data, (w,y), 0.0)))

    @variable(model, P_BAT_cap[b in BATS, y in YEARS] in
        MOI.Parameter(get(P_BAT_cap_data, (b,y), 0.0)))

    @variable(model, E_BAT_cap[b in BATS, y in YEARS] in
        MOI.Parameter(get(E_BAT_cap_data, (b,y), 0.0)))

    @variable(model, pg[i in Gens, t in 1:T, d in days_subset, y in YEARS] in
        MOI.Parameter(get(pg_data, (i,t,d,y), 0.0)))


    # VARIABLES
    @variable(model, va[i in keys(ref[:bus]), t=1:T, d in days_subset, y in YEARS])

    @variable(model,
        -ref[:branch][l]["rate_a"] <=
        p_rt[(l,i,j) in ref[:arcs_from], t=1:T, d in days_subset, y in YEARS] <=
        ref[:branch][l]["rate_a"]
    )

    @variable(model, p_shed_rt[i in keys(ref[:bus]), t=1:T, d in days_subset, y in YEARS] >= 0)
    @variable(model, p_cur_rt[i in keys(ref[:bus]),  t=1:T, d in days_subset, y in YEARS] >= 0)

    @variable(model, r_up[i in Gens,   t=1:T, d in days_subset, y in YEARS] >= 0)
    @variable(model, r_down[i in Gens, t=1:T, d in days_subset, y in YEARS] >= 0)

    @variable(model, e_rt[b in BATS, t=1:T, d in days_subset, y in YEARS] >= 0)
    @variable(model, P_BAT_rt[b in BATS, t=1:T, d in days_subset, y in YEARS])
    @variable(model, windpower_rt[w in RENW, t=1:T, d in days_subset, y in YEARS]>= 0)


    # EXISTING LINE FLOWS
    for l in EXISTLINES, d in days_subset, t in 1:T, y in YEARS
        br = ref[:branch][l]
        f = br["f_bus"]; to = br["t_bus"]
        @constraint(model,
            p_rt[(l,f,to),t,d,y] ==
            -B_line[l] * (va[f,t,d,y] - va[to,t,d,y])
        )
    end

    # SIGNED FLOW
    p_expr = Dict(
        ((l,i,j),t,d,y) => +p_rt[(l,i,j),t,d,y]
        for (l,i,j) in ref[:arcs_from], t in 1:T, d in days_subset, y in YEARS
    )

    p_expr = merge(
        p_expr,
        Dict(
            ((l,j,i),t,d,y) => -p_rt[(l,i,j),t,d,y]
            for (l,i,j) in ref[:arcs_from], t in 1:T, d in days_subset, y in YEARS
        )
    )


    # CANDIDATE LINES WITH GATING
    for l in CandLines, d in days_subset, t in 1:T, y in YEARS
        br = ref[:branch][l]
        f = br["f_bus"]; to = br["t_bus"]

        @constraint(model,
            p_rt[(l,f,to),t,d,y] ==
            -B_line[l] * (va[f,t,d,y] - va[to,t,d,y])
        )

        @constraint(model,  p_rt[(l,f,to),t,d,y] <=  f_cand[l,y])
        @constraint(model, -p_rt[(l,f,to),t,d,y] <=  f_cand[l,y])
    end


    # BATTERY DYNAMICS
    @variable(model, P_DIS_rt[b in BATS, t in 1:T, d in days_subset, y in YEARS] >= 0)
    @variable(model, P_CH_rt[b in BATS, t in 1:T, d in days_subset, y in YEARS] >= 0)

    
    for b in BATS, d in days_subset, t=1:T, y in YEARS
        @constraint(model, P_BAT_rt[b,t,d,y] <=  P_BAT_cap[b,y])
        @constraint(model, P_BAT_rt[b,t,d,y] >= -P_BAT_cap[b,y])
        @constraint(model, e_rt[b,t,d,y]    <=  E_BAT_cap[b,y])
        @constraint(model, P_BAT_rt[b,t,d,y] == P_DIS_rt[b,t,d,y] - P_CH_rt[b,t,d,y])
    end

    for b in BATS, d in days_subset, t=2:T, y in YEARS
        @constraint(model,
            e_rt[b,t,d,y] == e_rt[b,t-1,d,y] - P_BAT_rt[b,t,d,y]
        )
    end

    @constraint(model, [b in BATS, d in days_subset, y in YEARS],
        e_rt[b,1,d,y] == 0.5 * E_BAT_cap[b,y])

    @constraint(model, [b in BATS, d in days_subset, y in YEARS],
        e_rt[b,T,d,y] == 0.5 * E_BAT_cap[b,y])
   


    # wind

for g in RENW,  y in YEARS, d in days_subset, t in 1:T
    bus = bus_of_wind[g]
    @constraint(model,
        windpower_rt[g,t,d,y] <= (pwind_real[bus][t][d][y] / 400)* p_wind_cap[g,y]
    )
    @constraint(model, 0 <= windpower_rt[g,t,d,y])
end

    
    # NODAL BALANCE
    for y in YEARS, d in days_subset, t in 1:T
        for (i, _) in ref[:bus]

            wind_term = sum(
                windpower_rt[w,t,d,y]
                for w in RENW if bus_of_wind[w] == i;
                init = 0.0
            )

            @constraint(model,
                # flows
                sum(p_expr[a,t,d,y] for a in ref[:bus_arcs][i]; init = 0.0) ==
                sum(pg[g,t,d,y] + r_up[g,t,d,y] - r_down[g,t,d,y]
                    for g in ref[:bus_gens][i]; init = 0.0) +
                # battery
                sum(P_BAT_rt[b,t,d,y]
                    for b in BATS if bus_of_bat[b] == i; init = 0.0) -
                # loads
                (loadyearly[y] * (loadd[d][t][i][y]/100) + loadyearly[y] * net_load_fixed[d][t][i]) +
                # wind
                wind_term +
               # shed & curtailment
                p_shed_rt[i,t,d,y] 
            )
        end
    end


    # GENERATION LIMITS
    for g in CandGens, d in days_subset, t=1:T, y in YEARS
        @constraint(model,
            pg[g,t,d,y] + r_up[g,t,d,y] - r_down[g,t,d,y] <= p_new[g,y]
        )
    @constraint(model,
            pg[g,t,d,y] + r_up[g,t,d,y] - r_down[g,t,d,y] >= 0
        )

    end

    for g in EXISTGENS, d in days_subset, t=1:T, y in YEARS
        Pmax = ref[:gen][g]["pmax"]
        @constraint(model,
            pg[g,t,d,y] + r_up[g,t,d,y] - r_down[g,t,d,y] <= Pmax
        )
    @constraint(model,
            pg[g,t,d,y] + r_up[g,t,d,y] - r_down[g,t,d,y] >= 0
        )
    end

    #ramp
    
   for g in CandGens, d in days_subset, t in 2:T, y in YEARS
    @constraint(model, pg[g,t,d,y] + r_up[g,t,d,y] - r_down[g,t,d,y] - (pg[g,t-1,d,y] + r_up[g,t-1,d,y] - r_down[g,t-1,d,y]) <= 0.8* p_new[g,y])
    @constraint(model, pg[g,t,d,y] + r_up[g,t,d,y] - r_down[g,t,d,y] - (pg[g,t+1,d,y] + r_up[g,t+1,d,y] - r_down[g,t+1,d,y]) <= p_new[g,y])
   end

    for g in EXISTGENS, d in days_subset, y in YEARS, t in 2:T
    Pmax = ref[:gen][g]["pmax"]
      @constraint(model, pg[g,t,d,y] + r_up[g,t,d,y] - r_down[g,t,d,y] - (pg[g,t-1,d,y] + r_up[g,t-1,d,y] - r_down[g,t-1,d,y]) <=0.8 * Pmax)
      @constraint(model, pg[g,t,d,y] + r_up[g,t,d,y] - r_down[g,t,d,y] - (pg[g,t+1,d,y] + r_up[g,t+1,d,y] - r_down[g,t+1,d,y]) <= Pmax)
    end



    for (i,_) in ref[:bus], d in days_subset, t in 1:T, y in YEARS
       # @constraint(model, 0 <= p_shed_da[i,t,d,y])
        @constraint(model, p_shed_rt[i,t,d,y] <= loadyearly[y] * (loadd[d][t][i][y]/100) + loadyearly[y] * net_load_fixed[d][t][i])
    end
    

for y in YEARS, d in days_subset, t in 1:T, (i,_) in ref[:bus]
    gens_at_bus = [g for g in RENW if bus_of_wind[g] == i]

    if isempty(gens_at_bus)
        @constraint(model, p_cur_rt[i,t,d,y] == 0)
    else
        @constraint(model,
            p_cur_rt[i,t,d,y] ==
            sum((pwind_real[i][t][d][y] / 400) * p_wind_cap[g,y] - windpower_rt[g,t,d,y]
                for g in gens_at_bus)
        )
    end
end

    DF = Dict(y => 1 / (1 + r)^y for y in YEARS)
    add_inner_objective!(
        model,
        ref, BATS,
        TotalYears, T, days_subset, ALLGENS, DF,
        r_cost_up, r_cost_dn,
        c_shed_prime, c_cur_prime,
        battery_data,
        r_up, r_down,
        p_shed_rt, p_cur_rt,
        P_BAT_rt,  
        windpower_rt,
        P_DIS_rt,
        P_CH_rt,
    )

    return model
end


# Helpers for stacking vars
function flatten_vars(container)
    v = VariableRef[]
    for x in container
        push!(v, x)
    end
    return v
end

"Collect ALL middle (day-ahead) variables in the SAME ORDER used in J_pg."
function collect_DA_vars_flat(MM::Model)
    v = VariableRef[]
    if haskey(MM, :pg)
        append!(v, flatten_vars(MM[:pg]))
    end
    return v
end

"Collect ALL inner (real-time) variables in the SAME ORDER used in J_P_total."
function collect_RT_vars_flat(IM::Model)
    v = VariableRef[]

    if haskey(IM, :P_BAT_rt)
        append!(v, flatten_vars(IM[:P_BAT_rt]))
    end
    if haskey(IM, :e_rt)
        append!(v, flatten_vars(IM[:e_rt]))
    end
    if haskey(IM, :r_up)
        append!(v, flatten_vars(IM[:r_up]))
    end
    if haskey(IM, :r_down)
        append!(v, flatten_vars(IM[:r_down]))
    end
    if haskey(IM, :p_shed_rt)
        append!(v, flatten_vars(IM[:p_shed_rt]))
    end
    if haskey(IM, :p_cur_rt)
        append!(v, flatten_vars(IM[:p_cur_rt]))
    end
    if haskey(IM, :va)
        append!(v, flatten_vars(IM[:va]))
    end
    if haskey(IM, :p_rt)
        append!(v, flatten_vars(IM[:p_rt]))
    end
    if haskey(IM, :windpower_rt)
        append!(v, flatten_vars(IM[:windpower_rt]))
    end
    if haskey(IM, :P_DIS_rt)
        append!(v, flatten_vars(IM[:P_DIS_rt]))
    end
    if haskey(IM, :P_CH_rt)
        append!(v, flatten_vars(IM[:P_CH_rt]))
    end

    
    return v
end



# . Flatten JuMP containers
function flatten_vars(x)
    out = VariableRef[]
    if x isa JuMP.VariableRef
        push!(out, x)
    else
        for v in values(x)
            append!(out, flatten_vars(v))
        end
    end
    return out
end

#  θ = [p_new; f_cand; p_wind_cap; P_BAT_cap; E_BAT_cap]
function collect_theta(
    model::Model,
    CandGens, CandLines, RENW, BATS, TotalYears,
)
    YEARS = 1:TotalYears
    θ = VariableRef[]
    # p_new[g,y]
    for g in CandGens, y in YEARS
        push!(θ, model[:p_new][g,y])
    end
    # f_cand[l,y]
    for l in CandLines, y in YEARS
        push!(θ, model[:f_cand][l,y])
    end
    # p_wind_cap[w,y]
    for w in RENW, y in YEARS
        push!(θ, model[:p_wind_cap][w,y])
    end
    # P_BAT_cap[b,y]
    for b in BATS, y in YEARS
        push!(θ, model[:P_BAT_cap][b,y])
    end
    # E_BAT_cap[b,y]
    for b in BATS, y in YEARS
        push!(θ, model[:E_BAT_cap][b,y])
    end
    return θ
end

# θ_names (same ordering as collect_theta)
function theta_labels(
    CandGens, CandLines, RENW, BATS, TotalYears,
)
    YEARS = 1:TotalYears
    labels = String[]
    # p_new[g,y]
    for g in CandGens, y in YEARS
        push!(labels, "p_new[$g,$y]")
    end
    # f_cand[l,y]
    for l in CandLines, y in YEARS
        push!(labels, "f_cand[$l,$y]")
    end
    # p_wind_cap[w,y]
    for w in RENW, y in YEARS
        push!(labels, "p_wind_cap[$w,$y]")
    end
    # P_BAT_cap[b,y]
    for b in BATS, y in YEARS
        push!(labels, "P_BAT_cap[$b,$y]")
    end
    # E_BAT_cap[b,y]
    for b in BATS, y in YEARS
        push!(labels, "E_BAT_cap[$b,$y]")
    end
    return labels
end

#pg "parameters" inside INNER model
# indexed by (g,t,d,y) using pg_data.
function collect_pg_params(
    IM::Model,
    Gens, T, DD, TotalYears; days_subset
)
    YEARS = 1:TotalYears

    if !haskey(IM, :pg)
        error("Inner model has no :pg_param container.")
    end

    pg_params = VariableRef[]
    for g in Gens, t in 1:T, d in days_subset, y in YEARS
        push!(pg_params, IM[:pg][g,t,d,y])
    end
    return pg_params
end


function differentiate_trilevel_optionB_blocked(
    ref, CandGens, CandLines, RENW, BATS,
    TotalYears, T,DD, r, eta,
    cand_wind_data, battery_data,
    ref_data, c_g_fix_id, c_g_var_id,
    c_l_fix_id, c_l_var_id,
    gen_var_cost, r_cost_up, r_cost_dn,
    loadyearly, loadd, net_load, net_load_fixed,
    pwind, pwind_real, bus_of_wind, bus_of_bat,
    ALLGENS, ExistGens, ExistLines, Gens,
    B_line, CandLinesDict,
    c_shed_prime, c_cur_prime;
    p_new_data,
    f_cand_data,
    p_wind_cap_data,
    P_BAT_cap_data,
    E_BAT_cap_data,
    pg_data,
    days_subset::Vector{Int} = collect(DD), 
)

    YEARS = 1:TotalYears
    DF = Dict(y => 1 / (1 + r)^y for y in YEARS)

    # Build middle (day-ahead) model
    
    println("Building middle model...")
    MM  = build_middle_model(
    ref, CandGens, CandLines, RENW, BATS,
    TotalYears, T, DD, DF, r, eta,            
    cand_wind_data, battery_data,
    ref_data, c_g_fix_id, c_g_var_id,
    c_l_fix_id, c_l_var_id,
    gen_var_cost, r_cost_up, r_cost_dn,
    loadyearly, loadd, net_load, net_load_fixed,
    pwind, pwind_real, bus_of_wind, bus_of_bat,
    ALLGENS, ExistGens, ExistLines, Gens,
    B_line, CandLinesDict,
    c_shed_prime, c_cur_prime;
    p_new_data      = p_new_data,
    f_cand_data     = f_cand_data,
    p_wind_cap_data = p_wind_cap_data,
    P_BAT_cap_data  = P_BAT_cap_data,
    E_BAT_cap_data  = E_BAT_cap_data,
    scenario_index  = 1,
    days_subset     = days_subset,  
)
    # Build inner (real-time) model
    println("Building inner model...")
    IM = build_inner_model(
        ref, CandGens, CandLines, RENW, BATS,
        TotalYears, T, DD,
        r_cost_up, r_cost_dn,
        battery_data, ref_data,
        cand_wind_data,                   
        loadyearly,loadd, realload, net_load_fixed, pwind_real,
        bus_of_wind, bus_of_bat,
        ALLGENS, Gens, ExistGens,
        ExistLines, CandLinesDict,
        B_line, c_shed_prime, c_cur_prime,
        eta;                              
        p_new_data      = p_new_data,
        f_cand_data     = f_cand_data,
        p_wind_cap_data = p_wind_cap_data,
        P_BAT_cap_data  = P_BAT_cap_data,
        E_BAT_cap_data  = E_BAT_cap_data,
        pg_data         = pg_data,
        days_subset     = days_subset,
    )

    # Solve both models
    println("Solving base middle & inner models...")
    optimize!(MM)
    optimize!(IM)

    statM = termination_status(MM)
    statI = termination_status(IM)
    println("Middle status = $statM, Inner status = $statI")

    statM != MOI.OPTIMAL && error("Middle model not optimal: $statM")
    statI != MOI.OPTIMAL && @warn "Inner model not optimal; gradients may be inaccurate." statI

    # Collect θ in a consistent ordering
    
    θ_M     = collect_theta(MM, CandGens, CandLines, RENW, BATS, TotalYears)
    θ_I     = collect_theta(IM, CandGens, CandLines, RENW, BATS, TotalYears)
    θ_names = theta_labels(CandGens, CandLines, RENW, BATS, TotalYears)

    if length(θ_M) != length(θ_I)
        error("Parameter mismatch: θ_M and θ_I differ in length.")
    end
    nθ = length(θ_M)

    # 1) ALL DAY-AHEAD VARIABLES & J_DA_theta = d x_DA / dθ
    println("Collecting all day-ahead variables...")
    xDA_list = Any[]
    append!(xDA_list, flatten_vars(MM[:va]))
    append!(xDA_list, flatten_vars(MM[:pg]))
    append!(xDA_list, flatten_vars(MM[:p_da]))
    append!(xDA_list, flatten_vars(MM[:p_shed_da]))
    append!(xDA_list, flatten_vars(MM[:p_cur_da]))
    append!(xDA_list, flatten_vars(MM[:e_da]))
    append!(xDA_list, flatten_vars(MM[:P_BAT_da]))
    append!(xDA_list, flatten_vars(MM[:windpower_da]))
    append!(xDA_list, flatten_vars(MM[:P_DIS_da]))
    append!(xDA_list, flatten_vars(MM[:P_CH_da]))

    xDA_vec = Vector{VariableRef}(xDA_list)
    n_DA = length(xDA_vec)
    println("Total number of day-ahead variables = $n_DA")

    println("Computing J_DA_theta = d x_DA / d θ ...")
    J_DA_theta = zeros(Float64, n_DA, nθ)
    numeric_jacobian_forward!(J_DA_theta, MM, θ_M, xDA_vec)

    # 2) J_pg_theta = d pg (middle) / d θ
    haskey(MM, :pg) || error("Middle model has no :pg variable.")
    pg_MM_vec = flatten_vars(MM[:pg])  
    n_pg = length(pg_MM_vec)

    println("Computing J_pg_theta (size $(n_pg) x $(nθ)) ...")
    J_pg_theta = zeros(Float64, n_pg, nθ)
    numeric_jacobian_forward!(J_pg_theta, MM, θ_M, pg_MM_vec)

    # keep short name
    J_pg = J_pg_theta

    # 3) ALL INNER (REAL-TIME) VARIABLES
    println("Collecting all inner (real-time) variables...")
    xRT_list = Any[]
    append!(xRT_list, flatten_vars(IM[:P_BAT_rt]))
    append!(xRT_list, flatten_vars(IM[:e_rt]))
    append!(xRT_list, flatten_vars(IM[:r_up]))
    append!(xRT_list, flatten_vars(IM[:r_down]))
    append!(xRT_list, flatten_vars(IM[:p_shed_rt]))
    append!(xRT_list, flatten_vars(IM[:p_cur_rt]))
    append!(xRT_list, flatten_vars(IM[:va]))
    append!(xRT_list, flatten_vars(IM[:p_rt]))
    append!(xRT_list, flatten_vars(IM[:windpower_rt]))
    append!(xRT_list, flatten_vars(IM[:P_DIS_rt]))
    append!(xRT_list, flatten_vars(IM[:P_CH_rt]))

    xRT_vec = Vector{VariableRef}(xRT_list)
    n_RT = length(xRT_vec)
    println("Total number of real-time variables = $n_RT")

    # 3a) d xRT / d θ (direct)
    println("Computing J_xRT_theta_direct = d xRT / d θ ...")
    J_xRT_theta_direct = zeros(Float64, n_RT, nθ)
    numeric_jacobian_forward!(J_xRT_theta_direct, IM, θ_I, xRT_vec)

    # 3b) d xRT / d pg   (sensitivity wrt pg parameters)
    println("Collecting pg parameters inside inner model...")
    pg_IM_params = collect_pg_params(IM, Gens, T, DD, TotalYears; days_subset = days_subset)
    if length(pg_IM_params) != n_pg
        error("Mismatch: length(pg_IM_params) = $(length(pg_IM_params)) but n_pg = $n_pg")
    end

    println("Computing J_xRT_pg = d xRT / d pg ...")
    J_xRT_pg = zeros(Float64, n_RT, n_pg)
    numeric_jacobian_forward!(J_xRT_pg, IM, pg_IM_params, xRT_vec)

    # 3c) total d xRT / d θ via chain rule
    println("Computing J_xRT_theta_total ...")
    J_xRT_theta_total = J_xRT_theta_direct + J_xRT_pg * J_pg_theta

    J_P_total = J_xRT_theta_total

    # 4) Save and return
    println("Saving jacobians_optionB_blocked.jld2 ...")
    JLD2.jldsave("jacobians_optionB_blocked.jld2";
        theta_names        = θ_names,
        J_pg_theta         = J_pg_theta,
        J_DA_theta         = J_DA_theta,
        J_xRT_theta_direct = J_xRT_theta_direct,
        J_xRT_pg           = J_xRT_pg,
        J_xRT_theta_total  = J_xRT_theta_total,
    )

    println("Done Part 4.")
    return (θ_names, J_pg, J_P_total, J_DA_theta, J_xRT_pg, MM, IM)
end


function differentiate_trilevel_optionB_blocked_vjp(
    ref, CandGens, CandLines, RENW, BATS,
    TotalYears, T,DD, r, eta,
    cand_wind_data, battery_data,
    ref_data, c_g_fix_id, c_g_var_id,
    c_l_fix_id, c_l_var_id,
    gen_var_cost, r_cost_up, r_cost_dn,
    loadyearly, loadd, net_load, net_load_fixed,
    pwind, pwind_real, bus_of_wind, bus_of_bat,
    ALLGENS, ExistGens, ExistLines, Gens,
    B_line, CandLinesDict,
    c_shed_prime, c_cur_prime;
    p_new_data,
    f_cand_data,
    p_wind_cap_data,
    P_BAT_cap_data,
    E_BAT_cap_data,
    pg_data,
    days_subset::Vector{Int} = collect(DD),
)

    YEARS = 1:TotalYears
    DF = Dict(y => 1 / (1 + r)^y for y in YEARS)

    # --- Build models (both are DiffOpt models) ---
    MM = build_middle_model(
        ref, CandGens, CandLines, RENW, BATS,
        TotalYears, T, DD, DF, r, eta,
        cand_wind_data, battery_data,
        ref_data, c_g_fix_id, c_g_var_id,
        c_l_fix_id, c_l_var_id,
        gen_var_cost, r_cost_up, r_cost_dn,
        loadyearly, loadd, net_load, net_load_fixed,
        pwind, pwind_real, bus_of_wind, bus_of_bat,
        ALLGENS, ExistGens, ExistLines, Gens,
        B_line, CandLinesDict,
        c_shed_prime, c_cur_prime;
        p_new_data      = p_new_data,
        f_cand_data     = f_cand_data,
        p_wind_cap_data = p_wind_cap_data,
        P_BAT_cap_data  = P_BAT_cap_data,
        E_BAT_cap_data  = E_BAT_cap_data,
        scenario_index  = 1,
        days_subset     = days_subset,
    )


    optimize!(MM)

    for g in Gens, t in 1:T, d in days_subset, y in 1:TotalYears
    pg_data[(g,t,d,y)] = value(MM[:pg][g,t,d,y])
    end
    statM = termination_status(MM)
    statM != MOI.OPTIMAL && error("Middle model not optimal: $statM")
    
    IM = build_inner_model(
        ref, CandGens, CandLines, RENW, BATS,
        TotalYears, T, DD,
        r_cost_up, r_cost_dn,
        battery_data, ref_data,
        cand_wind_data,
        loadyearly, loadd, realload, net_load_fixed, pwind_real,
        bus_of_wind, bus_of_bat,
        ALLGENS, Gens, ExistGens,
        ExistLines, CandLinesDict,
        B_line, c_shed_prime, c_cur_prime,
        eta;
        p_new_data      = p_new_data,
        f_cand_data     = f_cand_data,
        p_wind_cap_data = p_wind_cap_data,
        P_BAT_cap_data  = P_BAT_cap_data,
        E_BAT_cap_data  = E_BAT_cap_data,
        pg_data         = pg_data,
        days_subset     = days_subset,
    )


    optimize!(IM)

    statI = termination_status(IM)
    statI != MOI.OPTIMAL && @warn "Inner model not optimal; gradients may be inaccurate." statI

    # --- Parameter vectors (θ) ---
    θ_M     = collect_theta(MM, CandGens, CandLines, RENW, BATS, TotalYears)
    θ_I     = collect_theta(IM, CandGens, CandLines, RENW, BATS, TotalYears)
    θ_names = theta_labels(CandGens, CandLines, RENW, BATS, TotalYears)

    # --- Day-ahead variable list x_DA (same ordering used for costs) ---
    xDA_list = Any[]
    append!(xDA_list, flatten_vars(MM[:va]))
    append!(xDA_list, flatten_vars(MM[:pg]))
    append!(xDA_list, flatten_vars(MM[:p_da]))
    append!(xDA_list, flatten_vars(MM[:p_shed_da]))
    append!(xDA_list, flatten_vars(MM[:p_cur_da]))
    append!(xDA_list, flatten_vars(MM[:e_da]))
    append!(xDA_list, flatten_vars(MM[:P_BAT_da]))
    append!(xDA_list, flatten_vars(MM[:windpower_da]))
    append!(xDA_list, flatten_vars(MM[:P_DIS_da]))
    append!(xDA_list, flatten_vars(MM[:P_CH_da]))

    xDA_vec = Vector{VariableRef}(xDA_list)

    # --- Inner variable list x_RT (same ordering used for costs) ---
    xRT_list = Any[]
    append!(xRT_list, flatten_vars(IM[:P_BAT_rt]))
    append!(xRT_list, flatten_vars(IM[:e_rt]))
    append!(xRT_list, flatten_vars(IM[:r_up]))
    append!(xRT_list, flatten_vars(IM[:r_down]))
    append!(xRT_list, flatten_vars(IM[:p_shed_rt]))
    append!(xRT_list, flatten_vars(IM[:p_cur_rt]))
    append!(xRT_list, flatten_vars(IM[:va]))
    append!(xRT_list, flatten_vars(IM[:p_rt]))
    append!(xRT_list, flatten_vars(IM[:windpower_rt]))
    append!(xRT_list, flatten_vars(IM[:P_DIS_rt]))
    append!(xRT_list, flatten_vars(IM[:P_CH_rt]))
    xRT_vec = Vector{VariableRef}(xRT_list)

    # --- DA pg vector for chaining ---
    pg_MM_vec = flatten_vars(MM[:pg])

    # --- pg "parameters" inside inner model (these are MOI.Parameter vars) ---
    pg_IM_params = collect_pg_params(IM, Gens, T, DD, TotalYears; days_subset = days_subset)

    return (DF, θ_names, θ_M, θ_I, xDA_vec, xRT_vec, pg_MM_vec, pg_IM_params, MM, IM)
end




function parse_indices4(v::JuMP.VariableRef)
    s = string(v)           
    i1 = findfirst('[', s)
    i2 = findlast(']', s)
    inside = s[i1+1 : i2-1]   
    return split(inside, ",")
end



function build_outer_cost_vectors(
    θ_names, J_DA, J_P_total, MM, IM, DF, ref,
    CandGens, CandLines, RENW, BATS, TotalYears,
    ALLGENS, gen_var_cost, r_cost_up, r_cost_dn,
    c_g_var_id, c_l_var_id, cand_wind_data, battery_data,
    c_shed_prime, c_cur_prime;
    days_subset::Vector{Int},
    current_penalty::Float64,
    annual_violation_flag::Dict{Int, Bool} 
)
    # (0) Sizes
    nθ   = length(θ_names)
    n_DA = size(J_DA, 1)
    n_RT = size(J_P_total, 1)

    day_filter(d) = d in days_subset
    nb = length(days_subset)

    # (1) c_theta – direct derivative wrt θ (investment costs)
    c_theta = zeros(nθ)

    for (k, name) in pairs(θ_names)
        if startswith(name, "p_new[")
            g_str, y_str = split(name[7:end-1], ",")
            g = parse(Int, g_str)
            y = parse(Int, y_str)
            c_theta[k] = DF[y] * c_g_var_id[g]

        elseif startswith(name, "f_cand[")
            l_str, y_str = split(name[8:end-1], ",")
            l = parse(Int, l_str)
            y = parse(Int, y_str)
            c_theta[k] = DF[y] * c_l_var_id[l]

        elseif startswith(name, "p_wind_cap[")
            inner = name[length("p_wind_cap[") + 1:end-1]
            g_str, y_str = split(inner, ",")
            g_str = strip(g_str)
            y = parse(Int, y_str)
            c_theta[k] = DF[y] * cand_wind_data[g_str].c_var

        elseif startswith(name, "P_BAT_cap[")
            inner = name[length("P_BAT_cap[") + 1:end-1]
            b_str, y_str = split(inner, ",")
            b_str = strip(b_str)
            y = parse(Int, y_str)
            c_theta[k] = 0.0

        elseif startswith(name, "E_BAT_cap[")
            inner = name[length("E_BAT_cap[") + 1:end-1]
            b_str, y_str = split(inner, ",")
            b_str = strip(b_str)
            y = parse(Int, y_str)
            c_theta[k] = DF[y] * battery_data[b_str].c_var_E
        end
    end

    # (2) c_DA
    c_DA_base    = zeros(n_DA)
    c_DA_penalty = zeros(n_DA)
    offset = 0

    #  va[i,t,d,y] : no explicit cost
    va_vec = flatten_vars(MM[:va])
    for v in va_vec
        offset += 1
    end

    # pg[g,t,d,y]
    pg_vec = flatten_vars(MM[:pg])
    for v in pg_vec
        idx  = parse_indices4(v)
        g_id = parse(Int, idx[1])
        d    = parse(Int, idx[3])
        y    = parse(Int, idx[4])
        offset += 1

        if day_filter(d)
            c_DA_base[offset] = DF[y] * gen_var_cost[g_id]

            if annual_violation_flag[y]
                c_DA_penalty[offset] = DF[y] * current_penalty * eta[y]
            end
        end
    end

    # p_da[(l,i,j),t,d,y] : no explicit cost
    p_da_vec = flatten_vars(MM[:p_da])
    for v in p_da_vec
        offset += 1
    end

    #p_shed_da[i,t,d,y]
    p_shed_vec = flatten_vars(MM[:p_shed_da])
    for v in p_shed_vec
        idx = parse_indices4(v)
        d   = parse(Int, idx[3])
        y   = parse(Int, idx[4])
        offset += 1
        c_DA_base[offset] = day_filter(d) ? DF[y] * c_shed_prime : 0.0
    end

    #p_cur_da[i,t,d,y]
    p_cur_vec = flatten_vars(MM[:p_cur_da])
    for v in p_cur_vec
        idx = parse_indices4(v)
        d   = parse(Int, idx[3])
        y   = parse(Int, idx[4])
        offset += 1
        c_DA_base[offset] = day_filter(d) ? DF[y] * c_cur_prime : 0.0
    end

    #  e_da[b,t,d,y] : no explicit cost
    e_da_vec = flatten_vars(MM[:e_da])
    for v in e_da_vec
        offset += 1
    end

    #  P_BAT_da[b,t,d,y] : no explicit cost
    P_BAT_da_vec = flatten_vars(MM[:P_BAT_da])
    for v in P_BAT_da_vec
        offset += 1
    end

    #  windpower_da[w,t,d,y] : no explicit cost
    windpower_da_vec = flatten_vars(MM[:windpower_da])
    for v in windpower_da_vec
        offset += 1
    end

    #  P_DIS_da[b,t,d,y]
    P_DIS_da_vec = flatten_vars(MM[:P_DIS_da])
    for v in P_DIS_da_vec
        idx   = parse_indices4(v)
        b_str = idx[1]
        d     = parse(Int, idx[3])
        y     = parse(Int, idx[4])
        offset += 1
        c_DA_base[offset] = day_filter(d) ? DF[y] * battery_data[b_str].c_opr_b : 0.0
    end

    #  P_CH_da[b,t,d,y]
    P_CH_da_vec = flatten_vars(MM[:P_CH_da])
    for v in P_CH_da_vec
        idx   = parse_indices4(v)
        b_str = idx[1]
        d     = parse(Int, idx[3])
        y     = parse(Int, idx[4])
        offset += 1
        c_DA_base[offset] = day_filter(d) ? DF[y] * battery_data[b_str].c_opr_b : 0.0
    end

    if offset != n_DA
        @warn "DA cost vector length (filled = $offset) does not match n_DA = $n_DA."
    end

    # (3) c_RT 
    c_RT_base    = zeros(n_RT)
    c_RT_penalty = zeros(n_RT)
    offset = 0

    #  P_BAT_rt[b,t,d,y] : no explicit cost
    P_rt = flatten_vars(IM[:P_BAT_rt])
    for v in P_rt
        offset += 1
    end

    # e_rt[b,t,d,y] : no explicit cost
    e_rt = flatten_vars(IM[:e_rt])
    for v in e_rt
        offset += 1
    end

    #  r_up[g,t,d,y]
    r_up_vec = flatten_vars(IM[:r_up])
    for v in r_up_vec
        idx  = parse_indices4(v)
        i_id = parse(Int, idx[1])
        d    = parse(Int, idx[3])
        y    = parse(Int, idx[4])
        offset += 1

        if day_filter(d)
            c_RT_base[offset] = DF[y] * r_cost_up[i_id]

            if annual_violation_flag[y]
                c_RT_penalty[offset] = DF[y] * current_penalty * eta[y]
            end
        end
    end

    # r_down[g,t,d,y]
    r_dn_vec = flatten_vars(IM[:r_down])
    for v in r_dn_vec
        idx  = parse_indices4(v)
        i_id = parse(Int, idx[1])
        d    = parse(Int, idx[3])
        y    = parse(Int, idx[4])
        offset += 1

        if day_filter(d)
            c_RT_base[offset] = -DF[y] * r_cost_dn[i_id]

            if annual_violation_flag[y]
                c_RT_penalty[offset] = -DF[y] * current_penalty * eta[y]
            end
        end
    end

    # p_shed_rt[i,t,d,y]
    p_shed_RT_vec = flatten_vars(IM[:p_shed_rt])
    for v in p_shed_RT_vec
        idx = parse_indices4(v)
        d   = parse(Int, idx[3])
        y   = parse(Int, idx[4])
        offset += 1
        c_RT_base[offset] = day_filter(d) ? DF[y] * c_shed_prime : 0.0
    end

    #  p_cur_rt[i,t,d,y]
    p_cur_RT_vec = flatten_vars(IM[:p_cur_rt])
    for v in p_cur_RT_vec
        idx = parse_indices4(v)
        d   = parse(Int, idx[3])
        y   = parse(Int, idx[4])
        offset += 1
        c_RT_base[offset] = day_filter(d) ? DF[y] * c_cur_prime : 0.0
    end

    #  va : no explicit cost
    va_RT_vec = flatten_vars(IM[:va])
    for v in va_RT_vec
        offset += 1
    end

    #  p_rt : no explicit cost
    p_rt_vec = flatten_vars(IM[:p_rt])
    for v in p_rt_vec
        offset += 1
    end

    #  windpower_rt[w,t,d,y]
    windpower_rt_vec = flatten_vars(IM[:windpower_rt])
    for v in windpower_rt_vec
        idx = parse_indices4(v)
        d   = parse(Int, idx[3])
        y   = parse(Int, idx[4])
        offset += 1

        if day_filter(d) && annual_violation_flag[y] > 0
            c_RT_penalty[offset] = DF[y] * current_penalty * eta[y] - DF[y] * current_penalty
        end
    end

    #  P_DIS_rt[b,t,d,y]
    P_DIS_rt_vec = flatten_vars(IM[:P_DIS_rt])
    for v in P_DIS_rt_vec
        idx   = parse_indices4(v)
        b_str = idx[1]
        d     = parse(Int, idx[3])
        y     = parse(Int, idx[4])
        offset += 1
        c_RT_base[offset] = day_filter(d) ? DF[y] * battery_data[b_str].c_opr_b : 0.0
    end

    # P_CH_rt[b,t,d,y]
    P_CH_rt_vec = flatten_vars(IM[:P_CH_rt])
    for v in P_CH_rt_vec
        idx   = parse_indices4(v)
        b_str = idx[1]
        d     = parse(Int, idx[3])
        y     = parse(Int, idx[4])
        offset += 1
        c_RT_base[offset] = day_filter(d) ? DF[y] * battery_data[b_str].c_opr_b : 0.0
    end

    if offset != n_RT
        @warn "RT cost vector length (filled = $offset) does not match n_RT = $n_RT."
    end

    
    op_weight = 365.0
    scale = 1.0
    c_theta .*= scale

    c_DA = op_weight .* (c_DA_base) .+ 5 .* c_DA_penalty
    c_RT = op_weight .* (c_RT_base) .+ 5 .* c_RT_penalty

    return c_theta, c_DA, c_RT
end





# θ helpers
function theta_from_data(
    θ_names::Vector{String},
    p_new_data,
    f_cand_data,
    p_wind_cap_data,
    P_BAT_cap_data,
    E_BAT_cap_data,
)
    θ_vec = zeros(Float64, length(θ_names))

    for (k, name) in pairs(θ_names)

        if startswith(name, "p_new[")
            g_str, y_str = split(name[7:end-1], ",")
            g = parse(Int, g_str)
            y = parse(Int, y_str)
            θ_vec[k] = get(p_new_data, (g, y), 0.0)

        elseif startswith(name, "f_cand[")
            l_str, y_str = split(name[8:end-1], ",")
            l = parse(Int, l_str)
            y = parse(Int, y_str)
            θ_vec[k] = get(f_cand_data, (l, y), 0.0)

        elseif startswith(name, "p_wind_cap[")
            inner       = name[length("p_wind_cap[") + 1:end-1]
            g_str, y_str = split(inner, ",")
            g_str = strip(g_str)
            y     = parse(Int, y_str)
            θ_vec[k] = get(p_wind_cap_data, (g_str, y), 0.0)

        elseif startswith(name, "P_BAT_cap[")
            inner        = name[length("P_BAT_cap[") + 1:end-1]
            b_str, y_str = split(inner, ",")
            b_str = strip(b_str)
            y     = parse(Int, y_str)
            θ_vec[k] = get(P_BAT_cap_data, (b_str, y), 0.0)

        elseif startswith(name, "E_BAT_cap[")
            inner        = name[length("E_BAT_cap[") + 1:end-1]
            b_str, y_str = split(inner, ",")
            b_str = strip(b_str)
            y     = parse(Int, y_str)
            θ_vec[k] = get(E_BAT_cap_data, (b_str, y), 0.0)
        end
    end

    return θ_vec
end


function proj_H!(
    θ_vec::Vector{Float64},
    θ_names::Vector{String},
    ref, CandGens, CandLines, RENW, BATS, TotalYears,
    cand_wind_data, battery_data,
)
    clamp_between(x, lo, hi) = (x < lo ? lo : (x > hi ? hi : x))

    # ---- 1. Box constraints ----
    for (k, name) in pairs(θ_names)
        val = θ_vec[k]

        if startswith(name, "p_new[")
            g, y = parse.(Int, split(name[7:end-1], ","))
            pmax = ref[:gen][g]["pmax"]
            θ_vec[k] = clamp_between(val, 0.0, pmax)

        elseif startswith(name, "f_cand[")
            l, y = parse.(Int, split(name[8:end-1], ","))
            F = ref[:branch][l]["rate_a"]
            θ_vec[k] = clamp_between(val, 0.0, F)

        elseif startswith(name, "p_wind_cap[")
            inner = name[length("p_wind_cap[") + 1:end-1]
            gstr, ystr = split(inner, ",")
            gstr = strip(gstr)
            y    = parse(Int, ystr)

            if haskey(cand_wind_data, gstr)
                pmax = cand_wind_data[gstr].pmax
                θ_vec[k] = clamp_between(val, 0.0, pmax)
            else
                @warn "No cand_wind_data entry for wind unit '$gstr'; only enforcing non-negativity."
                θ_vec[k] = max(val, 0.0)
            end

        elseif startswith(name, "P_BAT_cap[")
            inner = name[length("P_BAT_cap[") + 1:end-1]
            bstr, ystr = split(inner, ",")
            bstr = strip(bstr)
            y    = parse(Int, ystr)

            pmax = battery_data[bstr].p_dis
            θ_vec[k] = clamp_between(val, 0.0, pmax)

        elseif startswith(name, "E_BAT_cap[")
            inner = name[length("E_BAT_cap[") + 1:end-1]
            bstr, ystr = split(inner, ",")
            bstr = strip(bstr)
            y    = parse(Int, ystr)

            emax = battery_data[bstr].E
            θ_vec[k] = clamp_between(val, 0.0, emax)
        end
    end

    # ---- 2. Monotonicity in y ----
    if TotalYears >= 2
        for g in CandGens
            for y in 2:TotalYears
                idx_prev = findfirst(==("p_new[$g,$(y-1)]"), θ_names)
                idx_curr = findfirst(==("p_new[$g,$y]"), θ_names)
                if idx_prev !== nothing && idx_curr !== nothing
                    θ_vec[idx_curr] = max(θ_vec[idx_curr], θ_vec[idx_prev])
                end
            end
        end

        for l in CandLines
            for y in 2:TotalYears
                idx_prev = findfirst(==("f_cand[$l,$(y-1)]"), θ_names)
                idx_curr = findfirst(==("f_cand[$l,$y]"), θ_names)
                if idx_prev !== nothing && idx_curr !== nothing
                    θ_vec[idx_curr] = max(θ_vec[idx_curr], θ_vec[idx_prev])
                end
            end
        end

        for g in RENW
            for y in 2:TotalYears
                idx_prev = findfirst(==("p_wind_cap[$g,$(y-1)]"), θ_names)
                idx_curr = findfirst(==("p_wind_cap[$g,$y]"), θ_names)
                if idx_prev !== nothing && idx_curr !== nothing
                    θ_vec[idx_curr] = max(θ_vec[idx_curr], θ_vec[idx_prev])
                end
            end
        end

        for b in BATS
            for y in 2:TotalYears
                idx_prev = findfirst(==("P_BAT_cap[$b,$(y-1)]"), θ_names)
                idx_curr = findfirst(==("P_BAT_cap[$b,$y]"), θ_names)
                if idx_prev !== nothing && idx_curr !== nothing
                    θ_vec[idx_curr] = max(θ_vec[idx_curr], θ_vec[idx_prev])
                end
            end
        end

        for b in BATS
            for y in 2:TotalYears
                idx_prev = findfirst(==("E_BAT_cap[$b,$(y-1)]"), θ_names)
                idx_curr = findfirst(==("E_BAT_cap[$b,$y]"), θ_names)
                if idx_prev !== nothing && idx_curr !== nothing
                    θ_vec[idx_curr] = max(θ_vec[idx_curr], θ_vec[idx_prev])
                end
            end
        end
    end

    # ---- 3. Battery duration E = 4 P ----
    for b in BATS
        for y in 1:TotalYears
            nameP = "P_BAT_cap[$b,$y]"
            nameE = "E_BAT_cap[$b,$y]"
            idxP  = findfirst(==(nameP), θ_names)
            idxE  = findfirst(==(nameE), θ_names)

            if idxP !== nothing && idxE !== nothing
                Pval = θ_vec[idxP]
                E_desired = 4.0 * Pval
                emax = battery_data[b].E
                θ_vec[idxE] = clamp_between(E_desired, 0.0, emax)
            end
        end
    end

    return θ_vec
end


function update_data_from_theta!(
    θ_vec::Vector{Float64},
    θ_names::Vector{String},
    p_new_data,
    f_cand_data,
    p_wind_cap_data,
    P_BAT_cap_data,
    E_BAT_cap_data,
)
    @assert length(θ_vec) == length(θ_names)

    for (k, name) in pairs(θ_names)
        val = θ_vec[k]

        if startswith(name, "p_new[")
            m = match(r"p_new\[(\d+),(\d+)\]", name)
            m === nothing && error("Could not parse θ-name '$name' as p_new[g,y].")
            g = parse(Int, m.captures[1])
            y = parse(Int, m.captures[2])
            p_new_data[(g, y)] = val

        elseif startswith(name, "f_cand[")
            m = match(r"f_cand\[(\d+),(\d+)\]", name)
            m === nothing && error("Could not parse θ-name '$name' as f_cand[l,y].")
            l = parse(Int, m.captures[1])
            y = parse(Int, m.captures[2])
            f_cand_data[(l, y)] = val

        elseif startswith(name, "p_wind_cap[")
            m = match(r"p_wind_cap\[(.+),(\d+)\]", name)
            m === nothing && error("Could not parse θ-name '$name' as p_wind_cap[w,y].")
            w = strip(m.captures[1])
            y = parse(Int, m.captures[2])
            p_wind_cap_data[(w, y)] = val

        elseif startswith(name, "P_BAT_cap[")
            m = match(r"P_BAT_cap\[(.+),(\d+)\]", name)
            m === nothing && error("Could not parse θ-name '$name' as P_BAT_cap[b,y].")
            b = strip(m.captures[1])
            y = parse(Int, m.captures[2])
            P_BAT_cap_data[(b, y)] = val

        elseif startswith(name, "E_BAT_cap[")
            m = match(r"E_BAT_cap\[(.+),(\d+)\]", name)
            m === nothing && error("Could not parse θ-name '$name' as E_BAT_cap[b,y].")
            b = strip(m.captures[1])
            y = parse(Int, m.captures[2])
            E_BAT_cap_data[(b, y)] = val
        end
    end

    return nothing
end

function investment_cost_from_data(
    ref, CandGens, CandLines, RENW, BATS,
    TotalYears, DF,
    c_g_fix_id, c_g_var_id,
    c_l_fix_id, c_l_var_id,
    cand_wind_data, battery_data,
    p_new_data, f_cand_data,
    p_wind_cap_data, P_BAT_cap_data, E_BAT_cap_data;
    scale = 1.0,
)
    YEARS = 1:TotalYears
    inv = 0.0
    for y in YEARS
        disc = DF[y]
        inv += disc * (
            # candidate thermal gens
            sum( c_g_var_id[g]*p_new_data[(g,y)]   for g in CandGens) +
            # candidate lines
            sum( c_l_var_id[l]*f_cand_data[(l,y)] for l in CandLines) +
            # batteries
            sum(battery_data[b].c_var_E * E_BAT_cap_data[(b,y)]  for b in BATS) +
            # candidate wind
            sum(
                cand_wind_data[w].c_var * p_wind_cap_data[(w,y)] for w in RENW)
        )
    end
    return scale * inv
end



# Gradient step + projection

In [ ]:

DF = Dict(y => 1 / (1 + r)^y for y in 1:TotalYears)

#Histories
outer_hist      = Float64[]
gradnorm_hist   = Float64[]
alpha_hist      = Float64[]
grad_hist       = Vector{Vector{Float64}}()
theta_hist      = Vector{Vector{Float64}}()

day_cost_est    = fill(NaN, TotalDays)
full_obj_hist   = Float64[]

ma_window       = 10
ma_outer_hist   = Float64[]
ma_grad_hist    = Float64[]
ma_iters        = Int[]


# True full-objective evaluation

p_new_hist      = Dict{Tuple{Int,Int}, Vector{Float64}}()
full_eval_every = 1

true_full_obj_hist = Float64[]
true_full_obj_iters = Int[]

true_full_penalized_obj_hist = Float64[]
true_full_penalized_obj_iters = Int[]



f_cand_hist     = Dict{Tuple{Int,Int}, Vector{Float64}}()
p_wind_cap_hist = Dict{Tuple{String,Int}, Vector{Float64}}()
P_BAT_cap_hist  = Dict{Tuple{String,Int}, Vector{Float64}}()
E_BAT_cap_hist  = Dict{Tuple{String,Int}, Vector{Float64}}()

YEARS = 1:TotalYears
for g in CandGens, y in YEARS
   p_new_hist[(g,y)] = Float64[]
end
for l in CandLines, y in YEARS
   f_cand_hist[(l,y)] = Float64[]
end
for w in RENW, y in YEARS
   p_wind_cap_hist[(w,y)] = Float64[]
end
for b in BATS, y in YEARS
   P_BAT_cap_hist[(b,y)] = Float64[]
   E_BAT_cap_hist[(b,y)] = Float64[]
end

#Batch size > 1
batch_size = min(5, length(DD))
max_iters = 250
day_cost_est = fill(NaN, TotalDays)

#Penalties 

avg_c_fix_wind = mean(wdata.c_fix for wdata in values(cand_wind_data))
#penalty_ren    = 500 * avg_c_fix_wind
#penalty_ren_rt = 500 * avg_c_fix_wind



# Initialize Moving Average Variables

violation_ma = Dict(y => 0.0 for y in 1:TotalYears)
beta_ema = 0.2  # Smoothing factor 
total_days_in_year = length(DD)



current_penalty = 5.0 * avg_c_fix_wind
penalty_lr = 0.5  # Learning rate for the penalty 




#star
stationarity_resid_hist = Float64[]
ma_stationarity_resid_hist = Float64[]

violation_batch_hist = Dict(y => Float64[] for y in 1:TotalYears)
violation_ema_hist   = Dict(y => Float64[] for y in 1:TotalYears)
penalty_hist         = Float64[]




for it in 1:max_iters
   println("\n=== OUTER ITERATION $it ===")
   global current_penalty 
   perm = randperm(length(DD))
   batch_days = DD[perm[1:batch_size]]

    println("  -> sampled days in this batch: ", collect(batch_days))

  #  Base investment cost at current θ
   inv_obj = investment_cost_from_data(
       ref, CandGens, CandLines, RENW, BATS,
       TotalYears, DF,
       c_g_fix_id, c_g_var_id,
       c_l_fix_id, c_l_var_id,
       cand_wind_data, battery_data,
       p_new_data, f_cand_data,
       p_wind_cap_data, P_BAT_cap_data, E_BAT_cap_data;
       scale = 1.0,
   )



    # 1. Decide to apply penalties based on the Moving Average
    annual_violation_flag = Dict(y => (violation_ma[y] > 0) for y in 1:TotalYears)
    
    # Accumulators for THIS batch's generation
    batch_total_gen_accum = Dict(y => 0.0 for y in 1:TotalYears)
    batch_renew_gen_accum = Dict(y => 0.0 for y in 1:TotalYears)

    
 #   Current θ
   θ_names = String[]
   θ_vec   = Float64[]

   # Batch accumulators
   grad_inv      = nothing
   batch_grad_op = nothing
   batch_op_cost = 0.0

  #  Solve EACH day separately
   for (j, d) in enumerate(batch_days)
       println("    >> solving scenario day d = $d")

   #    days_subset = [d]
        local days_subset = [d]  
       DF_d, θ_names_d, θ_M_d, θ_I_d, xDA_vec_d, xRT_vec_d, pg_MM_vec_d, pg_IM_params_d, MM, IM =
           differentiate_trilevel_optionB_blocked_vjp(
               ref, CandGens, CandLines, RENW, BATS,
               TotalYears, T, DD, r, eta,
               cand_wind_data, battery_data,
               ref_data, c_g_fix_id, c_g_var_id,
               c_l_fix_id, c_l_var_id,
               gen_var_cost, r_cost_up, r_cost_dn,
               loadyearly, loadd, net_load, net_load_fixed,
               pwind, pwind_real, bus_of_wind, bus_of_bat,
               ALLGENS, ExistGens, ExistLines, Gens,
               B_line, CandLines,
               c_shed_prime, c_cur_prime;
               p_new_data      = p_new_data,
               f_cand_data     = f_cand_data,
               p_wind_cap_data = p_wind_cap_data,
               P_BAT_cap_data  = P_BAT_cap_data,
               E_BAT_cap_data  = E_BAT_cap_data,
               pg_data         = pg_data,
               days_subset     = days_subset,
           )

       if j == 1
           θ_names = θ_names_d
           θ_vec = theta_from_data(
               θ_names,
               p_new_data,
               f_cand_data,
               p_wind_cap_data,
               P_BAT_cap_data,
               E_BAT_cap_data,
           )
       else
           θ_names_d == θ_names || error("θ_names changed across scenarios!")
       end

   #     Build cost vectors for THIS DAY ONLY
       nθ_local   = length(θ_names)
       J_DA_dummy = zeros(Float64, length(xDA_vec_d), nθ_local)
       J_RT_dummy = zeros(Float64, length(xRT_vec_d), nθ_local)

       # 2. Extract primal generation variables for THIS day
        for y in 1:TotalYears
            day_total_gen = sum(
                value(MM[:pg][g,t,d,y]) + value(IM[:r_up][g,t,d,y]) - value(IM[:r_down][g,t,d,y])
                for g in ALLGENS, t in 1:T
            )
            day_renew = sum(
                value(IM[:windpower_rt][w,t,d,y])
                for w in RENW, t in 1:T
            )
            batch_total_gen_accum[y] += day_total_gen
            batch_renew_gen_accum[y] += day_renew
        end

        # 3. Call the modified cost vector function 
        c_theta_d, c_DA_d, c_RT_d = build_outer_cost_vectors(
            θ_names, J_DA_dummy, J_RT_dummy, MM, IM, DF, ref,
            CandGens, CandLines, RENW, BATS, TotalYears, ALLGENS,
            gen_var_cost, r_cost_up, r_cost_dn, c_g_var_id, c_l_var_id,
            cand_wind_data, battery_data, c_shed_prime, c_cur_prime;
            days_subset           = days_subset,
            current_penalty           = current_penalty,
            annual_violation_flag = annual_violation_flag  
        )

    #    Save investment gradient ONCE
       if j == 1
           grad_inv = copy(c_theta_d)
       end

     #   Extract primal values for THIS DAY ONLY
       x_DA_value_d = value.(vcat(
           flatten_vars(MM[:va]),
           flatten_vars(MM[:pg]),
           flatten_vars(MM[:p_da]),
           flatten_vars(MM[:p_shed_da]),
           flatten_vars(MM[:p_cur_da]),
           flatten_vars(MM[:e_da]),
           flatten_vars(MM[:P_BAT_da]),
           flatten_vars(MM[:windpower_da]),
           flatten_vars(MM[:P_DIS_da]),
           flatten_vars(MM[:P_CH_da]),
       ))

       x_RT_value_d = value.(vcat(
           flatten_vars(IM[:P_BAT_rt]),
           flatten_vars(IM[:e_rt]),
           flatten_vars(IM[:r_up]),
           flatten_vars(IM[:r_down]),
           flatten_vars(IM[:p_shed_rt]),
           flatten_vars(IM[:p_cur_rt]),
           flatten_vars(IM[:va]),
           flatten_vars(IM[:p_rt]),
           flatten_vars(IM[:windpower_rt]),
           flatten_vars(IM[:P_DIS_rt]),
           flatten_vars(IM[:P_CH_rt]),
       ))

      #  Day operating cost only
       op_cost_d =
           dot(c_DA_d, x_DA_value_d) +
           dot(c_RT_d, x_RT_value_d)

       batch_op_cost += op_cost_d
       day_cost_est[d] = op_cost_d

       println("      day operating cost = ", op_cost_d)

      #  Gradient for THIS DAY ONLY
       # (1) DA contribution
       vjp_DA = diffopt_vjp_parameters!(MM, θ_M_d, xDA_vec_d, c_DA_d)

       # (2) RT contribution wrt [θ ; pg]
       θI_all = vcat(θ_I_d, pg_IM_params_d)
       vjp_inner_all = diffopt_vjp_parameters!(IM, θI_all, xRT_vec_d, c_RT_d, Val(:all))

       nθ_local_inner = length(θ_M_d)
       grad_theta_direct = vjp_inner_all[1:nθ_local_inner]
       grad_pg           = vjp_inner_all[nθ_local_inner+1:end]

       # (3) chain through pg
        length(grad_pg) == length(pg_MM_vec_d) || error("grad_pg length mismatch with middle pg vector.")
       vjp_pg_chain = diffopt_vjp_parameters!(MM, θ_M_d, pg_MM_vec_d, grad_pg)

       # Operating gradient for THIS DAY ONLY
       grad_op_d = vjp_DA .+ grad_theta_direct .+ vjp_pg_chain

       if batch_grad_op === nothing
           batch_grad_op = copy(grad_op_d)
       else
           batch_grad_op .+= grad_op_d
       end
   end



 # 4. Update the Moving Average using the batch's expected annual generation
    for y in 1:TotalYears
        # Average daily generation in this batch
        avg_daily_total = batch_total_gen_accum[y] / batch_size
        avg_daily_renew = batch_renew_gen_accum[y] / batch_size
        
        # Scale up to expected annual generation
        est_annual_total = avg_daily_total * total_days_in_year
        est_annual_renew = avg_daily_renew * total_days_in_year
        
        # Calculate expected annual violation using formula
        est_annual_violation = eta[y] * (est_annual_total + est_annual_renew) - est_annual_renew
        
        # Update Exponential Moving Average
        if it == 1
            violation_ma[y] = est_annual_violation
        else
            violation_ma[y] = (1 - beta_ema) * violation_ma[y] + beta_ema * est_annual_violation
        end


        #star
        push!(violation_batch_hist[y], est_annual_violation)
        push!(violation_ema_hist[y], violation_ma[y])	  
        
        println("Year $y: Expected Annual Violation = $est_annual_violation | Moving Avg = $(violation_ma[y])")


#  DUAL ASCENT: ADJUST PENALTY
        # Allow penalty to go up AND down, but never below 0
        current_penalty = max(0.0, current_penalty + penalty_lr * violation_ma[y])

	println("Year $y: Violation = $(violation_ma[y]) | Current Penalty = $current_penalty")


    end
    
 
   # Final outer objective:
   # inv +  average(day operating cost over batch)
   batch_avg_op = batch_op_cost / length(batch_days)
   outer_obj    = inv_obj +  batch_avg_op

   println("BATCH operating cost (sum over sampled days) = ", batch_op_cost)
   println("BATCH average day operating cost             = ", batch_avg_op)
   println("BATCH objective = inv +  avg(day op)    = ", outer_obj)

   push!(outer_hist, outer_obj)

   # Estimated full objective over sampled unique days
   valid_idx = findall(!isnan, day_cost_est)
   avg_daily = mean(day_cost_est[valid_idx])
   J_est     = inv_obj +  avg_daily
   push!(full_obj_hist, J_est)

   println("ESTIMATED full objective over $(length(valid_idx)) sampled days = ", J_est)


# True full objective and true full penalized objective
# evaluated over all days
if it == 1 || it % full_eval_every == 0 || it == max_iters

    true_full_op_cost = 0.0
    true_full_penalized_op_cost = 0.0

    full_total_gen_accum = Dict(y => 0.0 for y in 1:TotalYears)
    full_renew_gen_accum = Dict(y => 0.0 for y in 1:TotalYears)

    for d in DD
        days_subset = [d]

        DF_d, θ_names_d, θ_M_d, θ_I_d, xDA_vec_d, xRT_vec_d,
        pg_MM_vec_d, pg_IM_params_d, MM, IM =
            differentiate_trilevel_optionB_blocked_vjp(
                ref, CandGens, CandLines, RENW, BATS,
                TotalYears, T, DD, r, eta,
                cand_wind_data, battery_data,
                ref_data, c_g_fix_id, c_g_var_id,
                c_l_fix_id, c_l_var_id,
                gen_var_cost, r_cost_up, r_cost_dn,
                loadyearly, loadd, net_load, net_load_fixed,
                pwind, pwind_real, bus_of_wind, bus_of_bat,
                ALLGENS, ExistGens, ExistLines, Gens,
                B_line, CandLines,
                c_shed_prime, c_cur_prime;
                p_new_data      = p_new_data,
                f_cand_data     = f_cand_data,
                p_wind_cap_data = p_wind_cap_data,
                P_BAT_cap_data  = P_BAT_cap_data,
                E_BAT_cap_data  = E_BAT_cap_data,
                pg_data         = pg_data,
                days_subset     = days_subset,
            )

        nθ_local = length(θ_names_d)
        J_DA_dummy = zeros(Float64, length(xDA_vec_d), nθ_local)
        J_RT_dummy = zeros(Float64, length(xRT_vec_d), nθ_local)

        # Pure cost vectors
        _, pure_c_DA_d, pure_c_RT_d = build_outer_cost_vectors(
            θ_names_d, J_DA_dummy, J_RT_dummy, MM, IM, DF, ref,
            CandGens, CandLines, RENW, BATS, TotalYears, ALLGENS,
            gen_var_cost, r_cost_up, r_cost_dn, c_g_var_id, c_l_var_id,
            cand_wind_data, battery_data, c_shed_prime, c_cur_prime;
            days_subset           = days_subset,
            current_penalty       = 0.0,
            annual_violation_flag = Dict(y => false for y in 1:TotalYears)
        )

        # Penalized cost vectors using current penalty
        _, pen_c_DA_d, pen_c_RT_d = build_outer_cost_vectors(
            θ_names_d, J_DA_dummy, J_RT_dummy, MM, IM, DF, ref,
            CandGens, CandLines, RENW, BATS, TotalYears, ALLGENS,
            gen_var_cost, r_cost_up, r_cost_dn, c_g_var_id, c_l_var_id,
            cand_wind_data, battery_data, c_shed_prime, c_cur_prime;
            days_subset           = days_subset,
            current_penalty       = current_penalty,
            annual_violation_flag = Dict(y => (violation_ma[y] > 0) for y in 1:TotalYears)
        )

        x_DA_value_d = value.(vcat(
            flatten_vars(MM[:va]), flatten_vars(MM[:pg]), flatten_vars(MM[:p_da]),
            flatten_vars(MM[:p_shed_da]), flatten_vars(MM[:p_cur_da]), flatten_vars(MM[:e_da]),
            flatten_vars(MM[:P_BAT_da]), flatten_vars(MM[:windpower_da]),
            flatten_vars(MM[:P_DIS_da]), flatten_vars(MM[:P_CH_da]),
        ))

        x_RT_value_d = value.(vcat(
            flatten_vars(IM[:P_BAT_rt]), flatten_vars(IM[:e_rt]), flatten_vars(IM[:r_up]),
            flatten_vars(IM[:r_down]), flatten_vars(IM[:p_shed_rt]), flatten_vars(IM[:p_cur_rt]),
            flatten_vars(IM[:va]), flatten_vars(IM[:p_rt]), flatten_vars(IM[:windpower_rt]),
            flatten_vars(IM[:P_DIS_rt]), flatten_vars(IM[:P_CH_rt]),
        ))

        true_full_op_cost += dot(pure_c_DA_d, x_DA_value_d) + dot(pure_c_RT_d, x_RT_value_d)

        true_full_penalized_op_cost += dot(pen_c_DA_d, x_DA_value_d) + dot(pen_c_RT_d, x_RT_value_d)

        # Full renewable violation over all days
        for y in 1:TotalYears
            day_total_gen = sum(
                value(MM[:pg][g,t,d,y]) +
                value(IM[:r_up][g,t,d,y]) -
                value(IM[:r_down][g,t,d,y])
                for g in ALLGENS, t in 1:T
            )

            day_renew = sum(
                value(IM[:windpower_rt][w,t,d,y])
                for w in RENW, t in 1:T
            )

            full_total_gen_accum[y] += day_total_gen
            full_renew_gen_accum[y] += day_renew
        end
    end

    # Pure full objective
    true_avg_daily_op = true_full_op_cost / length(DD)
    true_full_obj = inv_obj + true_avg_daily_op

    push!(true_full_obj_hist, true_full_obj)
    push!(true_full_obj_iters, it)

    # Penalized full objective
    true_full_penalty_term = 0.0

    for y in 1:TotalYears
        full_annual_total = full_total_gen_accum[y] / length(DD) * total_days_in_year
        full_annual_renew = full_renew_gen_accum[y] / length(DD) * total_days_in_year

        full_violation_y =
            eta[y] * (full_annual_total + full_annual_renew) -
            full_annual_renew

        true_full_penalty_term += current_penalty * full_violation_y
    end

    true_full_penalized_obj = true_full_obj + true_full_penalty_term

    push!(true_full_penalized_obj_hist, true_full_penalized_obj)
    push!(true_full_penalized_obj_iters, it)

    println("TRUE full pure objective over all days      = ", true_full_obj)
    println("TRUE full penalty term over all days        = ", true_full_penalty_term)
    println("TRUE full penalized objective over all days = ", true_full_penalized_obj)
end

    #Final gradient:
   # grad_inv + 365 * average(day operating gradients)
   grad_θ = grad_inv .+  (batch_grad_op ./ length(batch_days))

   gradnorm = norm(grad_θ)
   println("‖grad_θ‖₂ = ", gradnorm)

   push!(gradnorm_hist, gradnorm)
   push!(grad_hist, copy(grad_θ))

   # Moving average stopping rule
   if it >= ma_window && it % ma_window == 0
       obj_ma  = mean(outer_hist[end-ma_window+1:end])
       grad_ma = mean(gradnorm_hist[end-ma_window+1:end])

       push!(ma_outer_hist, obj_ma)
       push!(ma_grad_hist,  grad_ma)
       push!(ma_iters,      it)

       println("  Moving avg over last $ma_window iters:")
       println("    obj MA  = $obj_ma")
       println("    grad MA = $grad_ma")

       if grad_ma < 1e-3
           println("Moving-average gradient small => stopping.")
           break
       end
   end

  α_it = 0.00004 

   println("Step size α = ", α_it)
   push!(alpha_hist, α_it)


  #  Gradient step + projection
   θ_trial = θ_vec .- α_it .* grad_θ

   proj_H!(
       θ_trial, θ_names,
       ref, CandGens, CandLines, RENW, BATS, TotalYears,
       cand_wind_data, battery_data,
   )


#star

# Projected stationarity residual
   stationarity_resid = norm(θ_vec .- θ_trial)
   push!(stationarity_resid_hist, stationarity_resid)

   println("Projected stationarity residual = ", stationarity_resid)

   push!(penalty_hist, current_penalty)


   update_data_from_theta!(
       θ_trial, θ_names,
       p_new_data,
       f_cand_data,
       p_wind_cap_data,
       P_BAT_cap_data,
       E_BAT_cap_data,
   )

  #  Store updated θ
   push!(theta_hist, copy(θ_trial))

   for g in CandGens, y in YEARS
       push!(p_new_hist[(g,y)], p_new_data[(g,y)])
   end
   for l in CandLines, y in YEARS
       push!(f_cand_hist[(l,y)], f_cand_data[(l,y)])
   end
   for w in RENW, y in YEARS
       push!(p_wind_cap_hist[(w,y)], p_wind_cap_data[(w,y)])
   end
   for b in BATS, y in YEARS
       push!(P_BAT_cap_hist[(b,y)], P_BAT_cap_data[(b,y)])
       push!(E_BAT_cap_hist[(b,y)], E_BAT_cap_data[(b,y)])
   end
end


# Plots

In [ ]:
avg_c_fix_wind = mean(wdata.c_fix for wdata in values(cand_wind_data))
penalty_ren    = 5 * avg_c_fix_wind
current_penalty = 5 * avg_c_fix_wind


isdir("plots4") || mkdir("plots4")

iters = 1:length(outer_hist)

#star
# Projected stationarity residual
if !isempty(stationarity_resid_hist)
    p_resid = plot(
        1:length(stationarity_resid_hist),
        stationarity_resid_hist;
        xlabel = "Outer iteration",
        ylabel = "Projected stationarity residual",
        yscale = :log10,
        marker = :o,
        title = "Projected Stationarity Residual",
        legend = false,
        grid = true,
    )
    savefig(p_resid, joinpath("plots4", "projected_stationarity_residual.png"))
end

# Moving-average projected stationarity residual
if !isempty(ma_stationarity_resid_hist)
    p_ma_resid = plot(
        ma_iters,
        ma_stationarity_resid_hist;
        xlabel = "Outer iteration",
        ylabel = "Moving-average projected residual",
        yscale = :log10,
        marker = :o,
        title = "Projected Stationarity Residual Moving Average",
        legend = false,
        grid = true,
    )
    savefig(p_ma_resid, joinpath("plots4", "projected_residual_moving_average.png"))
end

# Renewable violation: mini-batch estimate vs EMA
for y in 1:TotalYears
    if !isempty(violation_batch_hist[y])
        p_vio = plot(
            1:length(violation_batch_hist[y]),
            violation_batch_hist[y];
            xlabel = "Outer iteration",
            ylabel = "Violation (MW)",
            label = "Mini-batch estimate",
            lw = 1,
            alpha = 0.4,
            title = "Renewable Constraint Violation, Year $y",
            grid = true,
        )

        plot!(
            p_vio,
            1:length(violation_ema_hist[y]),
            violation_ema_hist[y];
            label = "EMA tracked violation",
            lw = 2.5,
        )

        hline!(p_vio, [0.0]; label = "Feasibility threshold", linestyle = :dash)

        savefig(p_vio, joinpath("plots4", "renewable_violation_year_$(y).png"))
    end
end

# Penalty trajectory
if !isempty(penalty_hist)
    p_penalty = plot(
        1:length(penalty_hist),
        penalty_hist;
        xlabel = "Outer iteration",
        ylabel = "Penalty multiplier",
        marker = :o,
        title = "Penalty Trajectory",
        legend = false,
        grid = true,
    )
    savefig(p_penalty, joinpath("plots4", "penalty_trajectory.png"))
end

# True full pure vs penalized objective over all days
if !isempty(true_full_obj_hist) && !isempty(true_full_penalized_obj_hist)

    p_true_compare = plot(
        true_full_obj_iters,
        true_full_obj_hist;
        xlabel = "Outer iteration",
        ylabel = "Objective value",
        label = "Full pure objective",
        marker = :o,
        lw = 2,
        title = "Full Objective over All Days",
        grid = true,
    )

    plot!(
        p_true_compare,
        true_full_penalized_obj_iters,
        true_full_penalized_obj_hist;
        label = "Full penalized objective",
        marker = :x,
        lw = 2,
        linestyle = :dash,
    )

    savefig(
        p_true_compare,
        joinpath("plots4", "true_full_pure_vs_penalized_objective.png")
    )
end


if !isempty(true_full_obj_hist)
    p_true_full = plot(
        true_full_obj_iters,
        true_full_obj_hist;
        xlabel = "Outer iteration",
        ylabel = "Full objective over all days",
        title = "True Full Objective Convergence",
        marker = :o,
        legend = false,
        grid = true,
    )

    savefig(p_true_full, joinpath("plots4", "true_full_objective_convergence.png"))
end


# Full objective per iteration
if !isempty(full_obj_hist)
    full_slice = full_obj_hist[1:length(iters)]
    mask       = .!isnan.(full_slice)
    if any(mask)
        p_full = plot(
            findall(mask), full_slice[mask];
            xlabel = "Outer iteration",
            ylabel = "Estimated full objective",
            title  = "C_inv(θ) + 1/DD * Σ_d C_DA+RT(θ,d)",
            marker = :o,
            legend = false,
        )
        savefig(p_full, joinpath("plots4", "full_objective_convergence.png"))
    end
end



# Epoch definition
epoch_size = 2   
N_iters    = length(outer_hist)
num_epochs = ceil(Int, N_iters / epoch_size)

epoch_iter(e) = min(e * epoch_size, N_iters)

epoch_ids = [epoch_iter(e) for e in 1:num_epochs]

# 1) Estimated full objective (sample at end of each epoch)
if !isempty(full_obj_hist)
    full_slice = full_obj_hist[1:N_iters]

    epoch_full_obj = Float64[]
    epoch_epochs   = Int[]

    for e in 1:num_epochs
        idx = epoch_iter(e)
        val = full_slice[idx]
        if !isnan(val)
            push!(epoch_full_obj, val)
            push!(epoch_epochs, e)
        end
    end

    if !isempty(epoch_epochs)
        p_full = plot(
            epoch_epochs, epoch_full_obj,
            xlabel = "Epoch (100 iterations)",
            ylabel = "Objective",
            title  = "",
            marker = :o,
            legend = false,
        )
        savefig(p_full, joinpath("plots4", "full_objective_convergence_epoch.png"))
    end
end


# Two stage test model

In [ ]:

# extract investment variables at last iteration (y = 1) 
y = 1  

p_new_star = Dict(g => p_new_data[(g, y)]      for g in CandGens)
f_cand_star = Dict(l => f_cand_data[(l, y)]    for l in CandLines)
p_wind_cap_star = Dict(w => p_wind_cap_data[(w, y)] for w in RENW)

P_BAT_cap_star = Dict(b => P_BAT_cap_data[(b, y)] for b in BATS)
E_BAT_cap_star = Dict(b => E_BAT_cap_data[(b, y)] for b in BATS)

println("p_new*:       ", p_new_star)
println("f_cand*:      ", f_cand_star)
println("p_wind_cap*:  ", p_wind_cap_star)
println("P_BAT_cap*:   ", P_BAT_cap_star)
println("E_BAT_cap*:   ", E_BAT_cap_star)


# DAY-AHEAD (DA) MODEL FOR A SINGLE DAY  
"""
    solve_DA_for_day(d0;
        T::Int = 24,
        y::Int = 1,
        p_new_star::Dict{Int,Float64},
        f_cand_star::Dict{Int,Float64},
        p_wind_cap_star::Dict{String,Float64},
        P_BAT_cap_star::Dict{String,Float64},
        E_BAT_cap_star::Dict{String,Float64},
        loadyearly_vec::Vector{Float64} = loadyearly)

Builds and solves the Day-Ahead (DA) problem for a single day `d0`.

Inputs:
  - d0                  : day index 
  - T                   : number of hours (default 24)
  - y                   : year index 
  - p_new_star[g]       : candidate gen capacities from investment model
  - f_cand_star[l]      : candidate line capacities from investment model
  - p_wind_cap_star[g]  : candidate wind capacities from investment model
  - P_BAT_cap_star[b]   : battery power capacities from investment model
  - E_BAT_cap_star[b]   : battery energy capacities from investment model
  - loadyearly_vec      : the loadyearly array (e.g. [1.4, 1.15, ...])

Uses:
  Global data structures:
    ref, Gens, CandGens, ExistGens, Lines, CandLines, ExistLines,
    B_line, BATS, RENW, bus_of_wind, bus_of_bat,
    net_load, net_load_fixed, loadd, pwind, pwind_real,
    battery_data, cand_wind_data,
    c_shed_prime, c_cur_prime, eta

Returns:
  - obj_DA::Float64
  - pg_DA::Dict{Tuple{Int,Int},Float64} where key = (g, t)
  - model_DA::Model
"""
function solve_DA_for_day(d0::Int;
    T::Int = 24,
    y::Int = 1,
    p_new_star::Dict{Int,Float64},
    f_cand_star::Dict{Int,Float64},
    p_wind_cap_star::Dict{String,Float64},
    P_BAT_cap_star::Dict{String,Float64},
    E_BAT_cap_star::Dict{String,Float64},
    loadyearly_vec::Vector{Float64} = loadyearly
)

    ly = loadyearly_vec[y]

    # ------------------ Build model ------------------
    model_DA = Model(Gurobi.Optimizer)

    # ====== Variables ======
    # Voltage angles at buses
    @variable(model_DA, va_prime[i in keys(ref[:bus]), t in 1:T])

    # Generation schedule
    @variable(model_DA, pg[g in Gens, t in 1:T])

    @variable(model_DA, p_new_DA[g in CandGens] >= 0)
    @variable(model_DA, f_cand_DA[l in CandLines])
    @variable(model_DA, p_wind_cap_DA[w in RENW] >= 0)
    @variable(model_DA, P_BAT_cap_DA[b in BATS] >= 0)
    @variable(model_DA, E_BAT_cap_DA[b in BATS] >= 0)

    # Fix investment variables to planning solution
    for g in CandGens
        fix(p_new_DA[g], p_new_star[g]; force = true)
    end
    for l in CandLines
        fix(f_cand_DA[l], f_cand_star[l]; force = true)
    end
    for w in RENW
        fix(p_wind_cap_DA[w], p_wind_cap_star[w]; force = true)
    end
    for b in BATS
        fix(P_BAT_cap_DA[b], P_BAT_cap_star[b]; force = true)
        fix(E_BAT_cap_DA[b], E_BAT_cap_star[b]; force = true)
    end

    # Battery DA operation
    @variable(model_DA, e_da[b in BATS, t in 1:T])
    @variable(model_DA, P_BAT_da[b in BATS, t in 1:T])

    # Load shedding and wind curtailment
    @variable(model_DA, p_shed_prime[i in keys(ref[:bus]), t in 1:T] >= 0)
    @variable(model_DA, p_cur_prime[i in keys(ref[:bus]), t in 1:T]  >= 0)
    #wind
    @variable(model_DA, windp_da[i in RENW, t in 1:T]  >= 0)


    
    # Line flows
    @variable(model_DA,
        -ref[:branch][l]["rate_a"] <=
        p_prime[(l,i,j) in ref[:arcs_from], t in 1:T] <=
        ref[:branch][l]["rate_a"]
    )

    p_expr_prime = Dict(((l, i, j), t) => 1.0 * p_prime[(l, i, j), t]
                        for (l, i, j) in ref[:arcs_from], t in 1:T)
    p_expr_prime = merge(p_expr_prime,
        Dict(((l, j, i), t) => -1.0 * p_prime[(l, i, j), t]
            for (l, i, j) in ref[:arcs_from], t in 1:T))

    # ====== Constraints ======

    # --- DC power flow ---
    for l in ExistLines, t in 1:T
        br = ref[:branch][l]
        f_bus = br["f_bus"]
        t_bus = br["t_bus"]
        f_idx = (l, f_bus, t_bus)
        @constraint(model_DA,
            p_prime[f_idx, t] == -B_line[l] * (va_prime[f_bus, t] - va_prime[t_bus, t])
        )
    end
    for l in CandLines, t in 1:T
        br = ref[:branch][l]
        f_bus = br["f_bus"]
        t_bus = br["t_bus"]
        f_idx = (l, f_bus, t_bus)
        @constraint(model_DA,
            p_prime[f_idx, t] == -B_line[l] * (va_prime[f_bus, t] - va_prime[t_bus, t])
        )
    end

    # --- Angle reference bus(es) ---
    for (i, _bus) in ref[:ref_buses], t in 1:T
        @constraint(model_DA, va_prime[i, t] == 0)
    end

    # --- Generation limits ---
    for g in CandGens, t in 1:T
        @constraint(model_DA, pg[g, t] <= p_new_star[g])
        @constraint(model_DA, 0 <= pg[g, t])
    end
    for g in ExistGens, t in 1:T
        @constraint(model_DA, pg[g, t] <= ref[:gen][g]["pmax"])
        @constraint(model_DA, 0 <= pg[g, t])
    end

    #ramp

   for g in CandGens, t in 2:T
       @constraint(model_DA, pg[g, t] - pg[g, t-1]<= 0.8 * p_new_star[g])
       @constraint(model_DA, pg[g,t-1] - pg[g,t] <= 0.8 * p_new_star[g])
   end


    for g in ExistGens, t in 2:T
          @constraint(model_DA, pg[g, t] - pg[g, t-1]<= 0.8*  ref[:gen][g]["pmax"] )
          @constraint(model_DA, pg[g,t-1] - pg[g,t] <= 0.8 *  ref[:gen][g]["pmax"] )
    end
   

    # candidate line gating (lower)
    for l in CandLines, t in 1:T
        br = ref[:branch][l]; f = br["f_bus"]; to = br["t_bus"]; f_idx = (l, f, to)
        @constraint(model_DA, p_prime[f_idx,t] <=  f_cand_DA[l])
        @constraint(model_DA, -f_cand_DA[l] <=  p_prime[f_idx,t])
    end
    # --- Battery DA constraints ---
    @variable(model_DA, P_DIS_da[b in BATS, t in 1:T] >= 0)
    @variable(model_DA, P_CH_da[b in BATS, t in 1:T] >= 0)
    for b in BATS, t in 1:T
        @constraint(model_DA, P_BAT_da[b,t] == P_DIS_da[b,t] - P_CH_da[b,t])
    end

    for b in BATS, t in 1:T
        @constraint(model_DA, e_da[b, t] <= E_BAT_cap_star[b])
        @constraint(model_DA, 0 <= e_da[b, t])

        @constraint(model_DA, P_BAT_da[b, t] <= P_BAT_cap_star[b])
        @constraint(model_DA, -P_BAT_cap_star[b] <= P_BAT_da[b, t])
    end

    for b in BATS, t in 2:T
        @constraint(model_DA, e_da[b, t] == e_da[b, t-1] - P_BAT_da[b, t])
    end

    @constraint(model_DA, [b in BATS],
        e_da[b, 1] == 0.5 * E_BAT_cap_star[b]
    )
    @constraint(model_DA, [b in BATS],
        e_da[b, T] == 0.5 * E_BAT_cap_star[b]
    )



for g in RENW, t in 1:T
    bus = bus_of_wind[g]
    @constraint(model_DA,
        windp_da[g,t] <= (pwind[bus][t][d0][y] / 400)* p_wind_cap_star[g]
    )
    @constraint(model_DA, 0 <= windp_da[g,t])
end
    

    # --- Nodal balance for this day d0 ---
    for (i, _bus) in ref[:bus], t in 1:T
        @constraint(model_DA,
            sum(p_expr_prime[a, t] for a in ref[:bus_arcs][i]) ==
            sum(pg[g, t] for g in ref[:bus_gens][i])
            + sum(P_BAT_da[b, t] for b in BATS if bus_of_bat[b] == i)
            -   (ly * (loadd[d0][t][i][y]/100) + ly * net_load_fixed[d0][t][i])
            + sum(windp_da[w,t] for w in RENW if bus_of_wind[w] == i)
            + p_shed_prime[i, t]
        )
    end


    for (i,_) in ref[:bus], t in 1:T
        @constraint(model_DA, p_shed_prime[i,t] <= ly * (loadd[d0][t][i][y]/100) + loadyearly[y] * net_load_fixed[d0][t][i])
    end

for t in 1:T, (i,_) in ref[:bus]
    gens_at_bus = [g for g in RENW if bus_of_wind[g] == i]

    if isempty(gens_at_bus)
        @constraint(model_DA, p_cur_prime[i,t] == 0)
    else
        @constraint(model_DA,
            p_cur_prime[i,t] ==
            sum((pwind[i][t][d0][y] / 400) * p_wind_cap_star[g] - windp_da[g,t]
                for g in gens_at_bus)
        )
    end
end

    @objective(model_DA, Min,
        sum(gen_var_cost[g] * pg[g, t] for g in Gens, t in 1:T)
        +
        sum(c_shed_prime * p_shed_prime[i, t] +
            c_cur_prime  * p_cur_prime[i, t]
            for i in keys(ref[:bus]), t in 1:T)
        +
        sum(battery_data[b].c_opr_b * (P_DIS_da[b, t] + P_CH_da[b,t])
            for b in BATS, t in 1:T)
    )

    # ------------------ Solve ------------------
    optimize!(model_DA)

    obj_DA = objective_value(model_DA)

    # Extract pg schedule for use in RT
    pg_DA = Dict{Tuple{Int,Int},Float64}()
    for g in Gens, t in 1:T
        pg_DA[(g, t)] = value(pg[g, t])
    end

    return obj_DA, pg_DA, model_DA
end



# REAL-TIME (RT) MODEL FOR A SINGLE DAY 
"""
    solve_RT_for_day(d0;
        T::Int = 24,
        y::Int = 1,
        p_new_star::Dict{Int,Float64},
        p_wind_cap_star::Dict{String,Float64},
        P_BAT_cap_star::Dict{String,Float64},
        E_BAT_cap_star::Dict{String,Float64},
        r_cost_up::Dict{Int,Float64},
        r_cost_dn::Dict{Int,Float64},
        pg_DA::Dict{Tuple{Int,Int},Float64},
        loadyearly_vec::Vector{Float64} = loadyearly)

Builds and solves the Real-Time (RT) problem for a single day `d0`.

Inputs:
  - d0                    : day index
  - T                     : number of hours (default 24)
  - y                     : year index
  - p_new_star[g]         : candidate gen capacities from investment model
  - p_wind_cap_star[w]    : candidate wind capacities from investment model
  - P_BAT_cap_star[b]     : battery power capacities from investment model
  - E_BAT_cap_star[b]     : battery energy capacities from investment model
  - r_cost_up[g]          : upward reserve cost for each generator
  - r_cost_dn[g]          : downward reserve cost for each generator
  - pg_DA[(g,t)]          : DA scheduled generation for each (g,t)
  - loadyearly_vec        : the loadyearly array

Uses global data:
  ref, Gens, CandGens, ExistGens, Lines, CandLines, ExistLines,
  B_line, BATS, RENW, bus_of_wind, bus_of_bat,
  realload, net_load_fixed, pwind_real,
  battery_data, cand_wind_data,
  c_shed_prime, c_cur_prime, eta

Returns:
  - obj_RT::Float64
  - model_RT::Model
"""
function solve_RT_for_day(d0::Int;
    T::Int = 24,
    y::Int = 1,
    p_new_star::Dict{Int,Float64},
    f_cand_star::Dict{Int,Float64},    
    p_wind_cap_star::Dict{String,Float64},
    P_BAT_cap_star::Dict{String,Float64},
    E_BAT_cap_star::Dict{String,Float64},
    r_cost_up::Dict{Int,Float64},
    r_cost_dn::Dict{Int,Float64},
    pg_DA::Dict{Tuple{Int,Int},Float64},
    loadyearly_vec::Vector{Float64} = loadyearly
)

    ly = loadyearly_vec[y]

    # ------------------ Build model ------------------
    model_RT = Model(Gurobi.Optimizer)

    # ================== VARIABLES ==================
    # Voltage angles
    @variable(model_RT, va_prime_blnc[i in keys(ref[:bus]), t in 1:T])

    # RT reserves
    @variable(model_RT, r_up[g in Gens, t in 1:T] >= 0)
    @variable(model_RT, r_down[g in Gens, t in 1:T] >= 0)

    # Copy of pg but fixed to DA results
    @variable(model_RT, pg_RT[g in Gens, t in 1:T])
    for g in Gens, t in 1:T
        fix(pg_RT[g, t], pg_DA[(g, t)]; force = true)
    end

    # Shedding / curtailment (RT)
    @variable(model_RT,
              p_shed_prime_blnc[i in keys(ref[:bus]), t in 1:T] >= 0)
    @variable(model_RT,
              p_cur_prime_blnc[i in keys(ref[:bus]), t in 1:T]  >= 0)

    # RT battery operation
    @variable(model_RT, e_rt[b in BATS, t in 1:T])
    @variable(model_RT, P_BAT_rt[b in BATS, t in 1:T])

    # Fix battery capacities to investment
    @variable(model_RT, P_BAT_cap_RT[b in BATS] >= 0)
    @variable(model_RT, E_BAT_cap_RT[b in BATS] >= 0)
    for b in BATS
        fix(P_BAT_cap_RT[b], P_BAT_cap_star[b]; force = true)
        fix(E_BAT_cap_RT[b], E_BAT_cap_star[b]; force = true)
    end

    @variable(model_RT, p_new_RT[g in CandGens] >= 0)
    @variable(model_RT, f_cand_RT[l in CandLines])

    @variable(model_RT, p_wind_cap_RT[w in RENW] >= 0)
    for g in CandGens
        fix(p_new_RT[g], p_new_star[g]; force = true)
    end
    for l in CandLines
        fix(f_cand_RT[l], f_cand_star[l]; force = true)
    end
    for w in RENW
        fix(p_wind_cap_RT[w], p_wind_cap_star[w]; force = true)
    end

    # Line flows
    @variable(model_RT,
        -ref[:branch][l]["rate_a"] <=
        p_prime_blnc[(l,i,j) in ref[:arcs_from], t in 1:T] <=
        ref[:branch][l]["rate_a"]
    )

    @variable(model_RT, windp_rt[i in RENW, t in 1:T]  >= 0)


    
    p_expr_prime_blnc =
        Dict(((l, i, j), t) => 1.0 * p_prime_blnc[(l, i, j), t]
             for (l, i, j) in ref[:arcs_from], t in 1:T)
    p_expr_prime_blnc = merge(p_expr_prime_blnc,
        Dict(((l, j, i), t) => -1.0 * p_prime_blnc[(l, i, j), t]
            for (l, i, j) in ref[:arcs_from], t in 1:T))

    # ================== CONSTRAINTS ==================

    # --- DC power flow ---
    for l in ExistLines, t in 1:T
        br = ref[:branch][l]
        f_bus = br["f_bus"]
        t_bus = br["t_bus"]
        f_idx = (l, f_bus, t_bus)
        @constraint(model_RT,
            p_prime_blnc[f_idx, t] ==
            -B_line[l] * (va_prime_blnc[f_bus, t] - va_prime_blnc[t_bus, t])
        )
    end
    for l in CandLines, t in 1:T
        br = ref[:branch][l]
        f_bus = br["f_bus"]
        t_bus = br["t_bus"]
        f_idx = (l, f_bus, t_bus)
        @constraint(model_RT,
            p_prime_blnc[f_idx, t] ==
            -B_line[l] * (va_prime_blnc[f_bus, t] - va_prime_blnc[t_bus, t])
        )
    end

    # --- Angle reference ---
    for (i, _bus) in ref[:ref_buses], t in 1:T
        @constraint(model_RT, va_prime_blnc[i, t] == 0)
    end


    for l in CandLines, t in 1:T
        br = ref[:branch][l]; f = br["f_bus"]; to = br["t_bus"]; f_idx = (l, f, to)
        @constraint(model_RT, p_prime_blnc[f_idx,t] <=  f_cand_RT[l])
        @constraint(model_RT, -f_cand_RT[l] <=  p_prime_blnc[f_idx,t])
    end

    
    # --- Reserve capacity limits (based on investment) ---
    for g in CandGens, t in 1:T
        @constraint(model_RT, r_up[g, t]   <= p_new_star[g])
        @constraint(model_RT, r_down[g, t] <= p_new_star[g])
    end
    for g in ExistGens, t in 1:T
        @constraint(model_RT, r_up[g, t]   <= ref[:gen][g]["pmax"])
        @constraint(model_RT, r_down[g, t] <= ref[:gen][g]["pmax"])
    end

    for g in CandGens, t in 1:T
        @constraint(model_RT, pg_RT[g, t] + r_up[g, t] - r_down[g, t]   <= p_new_star[g])
        @constraint(model_RT, pg_RT[g, t] + r_up[g, t] - r_down[g, t]   >= 0)
    end
    for g in ExistGens, t in 1:T
        @constraint(model_RT, pg_RT[g, t] + r_up[g, t] - r_down[g, t] <= ref[:gen][g]["pmax"])
        @constraint(model_RT, pg_RT[g, t] + r_up[g, t] - r_down[g, t]   >= 0)
    end

    

    #ramp
    for g in CandGens, t in 2:T
       @constraint(model_RT, pg_RT[g,t] + r_up[g,t] - r_down[g,t] - (pg_RT[g, t-1] + r_up[g,t-1] - r_down[g,t-1]) <= 0.8*  p_new_star[g])
       @constraint(model_RT, pg_RT[g,t-1] + r_up[g,t-1] - r_down[g,t-1] - (pg_RT[g,t] + r_up[g,t] - r_down[g,t]) <= 0.8* p_new_star[g])
   end

    for g in ExistGens, t in 2:T
        @constraint(model_RT, pg_RT[g,t] + r_up[g,t] - r_down[g,t] - (pg_RT[g, t-1] + r_up[g,t-1] - r_down[g,t-1]) <= 0.8 * ref[:gen][g]["pmax"])
        @constraint(model_RT, pg_RT[g,t-1] + r_up[g,t-1] - r_down[g,t-1] - (pg_RT[g,t] + r_up[g,t] - r_down[g,t]) <= 0.8 * ref[:gen][g]["pmax"])
    end


    

    # --- Battery RT constraints ---
    @variable(model_RT, P_DIS_rt[b in BATS, t in 1:T] >= 0)
    @variable(model_RT, P_CH_rt[b in BATS, t in 1:T] >= 0)
    for b in BATS, t in 1:T
        @constraint(model_RT, P_BAT_rt[b,t] == P_DIS_rt[b,t] - P_CH_rt[b,t])
    end
    
    for b in BATS, t in 1:T
        @constraint(model_RT, e_rt[b, t] <= E_BAT_cap_star[b])
        @constraint(model_RT, 0 <= e_rt[b, t])

        @constraint(model_RT, P_BAT_rt[b, t] <= P_BAT_cap_star[b])
        @constraint(model_RT, -P_BAT_cap_star[b] <= P_BAT_rt[b, t])
    end

    for b in BATS, t in 2:T
        @constraint(model_RT, e_rt[b, t] == e_rt[b, t-1] - P_BAT_rt[b, t])
    end

    @constraint(model_RT, [b in BATS],
        e_rt[b, 1] == 0.5 * E_BAT_cap_star[b]
    )
    @constraint(model_RT, [b in BATS],
        e_rt[b, T] == 0.5 * E_BAT_cap_star[b]
    )

for g in RENW, t in 1:T
    bus = bus_of_wind[g]
    @constraint(model_RT,
        windp_rt[g,t] <= (pwind_real[bus][t][d0][y] / 400) * p_wind_cap_star[g]
    )
    @constraint(model_RT, 0 <= windp_rt[g,t])
end



    
    # --- Nodal balance with RT redispatch ---
    for (i, _bus) in ref[:bus], t in 1:T
        @constraint(model_RT,
            sum(p_expr_prime_blnc[a, t] for a in ref[:bus_arcs][i]) ==
            sum(pg_RT[g, t] + r_up[g, t] - r_down[g, t]
                for g in ref[:bus_gens][i])
            + sum(P_BAT_rt[b, t] for b in BATS if bus_of_bat[b] == i)
            - (ly * (loadd[d0][t][i][y]/100) + ly * net_load_fixed[d0][t][i])
            + sum(windp_rt[w,t] for w in RENW if bus_of_wind[w] == i)
            + p_shed_prime_blnc[i, t] 
        )
    end

    for (i,_) in ref[:bus], t in 1:T
        @constraint(model_RT, p_shed_prime_blnc[i,t] <= ly * (loadd[d0][t][i][y]/100) + loadyearly[y] * net_load_fixed[d0][t][i])
    end
  
for t in 1:T, (i,_) in ref[:bus]
    gens_at_bus = [g for g in RENW if bus_of_wind[g] == i]

    if isempty(gens_at_bus)
        @constraint(model_RT, p_cur_prime_blnc[i,t] == 0)
    else
        @constraint(model_RT,
            p_cur_prime_blnc[i,t] ==
            sum((pwind_real[i][t][d0][y] / 400) * p_wind_cap_star[g] - windp_rt[g,t]
                for g in gens_at_bus)
        )
    end
end


    @objective(model_RT, Min,
        sum(r_cost_up[g] * r_up[g, t] - r_cost_dn[g] * r_down[g, t]
            for g in Gens, t in 1:T)
        +
        sum(c_shed_prime * p_shed_prime_blnc[i, t] +
            c_cur_prime  * p_cur_prime_blnc[i, t]
            for i in keys(ref[:bus]), t in 1:T)
        +
        sum(battery_data[b].c_opr_b * (P_DIS_rt[b, t] + P_CH_rt[b,t])
            for b in BATS, t in 1:T)
    )

    # ------------------ Solve ------------------
    optimize!(model_RT)

    rt_up_cost = value(sum(r_cost_up[g] * r_up[g,t] for g in Gens, t in 1:T))
rt_dn_credit = value(sum(r_cost_dn[g] * r_down[g,t] for g in Gens, t in 1:T))
rt_shed_cost = value(sum(c_shed_prime * p_shed_prime_blnc[i,t] for i in keys(ref[:bus]), t in 1:T))
rt_cur_cost = value(sum(c_cur_prime * p_cur_prime_blnc[i,t] for i in keys(ref[:bus]), t in 1:T))
rt_bat_cost = value(sum(battery_data[b].c_opr_b * (P_DIS_rt[b,t] + P_CH_rt[b,t]) for b in BATS, t in 1:T))

println("RT day $d0 up    = ", rt_up_cost)
println("RT day $d0 down  = ", rt_dn_credit)
println("RT day $d0 shed  = ", rt_shed_cost)
println("RT day $d0 cur   = ", rt_cur_cost)
println("RT day $d0 bat   = ", rt_bat_cost)
println("RT day $d0 sum   = ", rt_up_cost - rt_dn_credit + rt_shed_cost + rt_cur_cost + rt_bat_cost )



# ------------------ Post-process totals ------------------
wind_total = sum(value(windp_rt[g, t]) for g in RENW, t in 1:T)

fuel_total = sum(
    value(pg_RT[g, t] + r_up[g, t] - r_down[g, t])
    for g in Gens, t in 1:T
)

ratio = fuel_total > 0 ? wind_total / fuel_total  : NaN

obj_RT = objective_value(model_RT)

println("RT day $d0 ratio    = ", ratio)    
println("RT day $d0 fuel_total   = ", fuel_total)  
println("RT day $d0 wind_total   = ", wind_total)  
    
return obj_RT, wind_total, fuel_total, ratio
end


# OOS

In [ ]:

N_total_days = 731

# Days we already used
used_days = collect(1:2:TotalDays)  

# All possible days
all_days = collect(1:N_total_days)

# Remaining (unused) days
sampled_days = setdiff(all_days, used_days)
# sampled_days = used_days
#sampled_days = all_days


DD = length(sampled_days)

println("Sampled days = ", sampled_days)

DA_obj = zeros(Float64, DD)
pg_all_days = Vector{Dict{Tuple{Int,Int},Float64}}(undef, DD)

# DAY-AHEAD PROBLEM #
for (i, d0) in enumerate(sampled_days)
    println("=== DA for day $d0 (sample index $i) ===")
    obj_DA, pg_DA, model_DA = solve_DA_for_day(
        d0;
        p_new_star      = p_new_star,
        f_cand_star     = f_cand_star,
        p_wind_cap_star = p_wind_cap_star,
        P_BAT_cap_star  = P_BAT_cap_star,
        E_BAT_cap_star  = E_BAT_cap_star,
    )
    DA_obj[i] = obj_DA
    pg_all_days[i] = pg_DA
end

total_DA = sum(DA_obj)
println("Total DA cost over sampled days = ", total_DA)

# REAL-TIME PROBLEM #
RT_obj = zeros(Float64, DD)
wind_totals = zeros(Float64, DD)
fuel_totals = zeros(Float64, DD)
ratios = zeros(Float64, DD)

for (i, d0) in enumerate(sampled_days)
    println("=== RT for day $d0 (sample index $i) ===")

    pg_DA = pg_all_days[i]

    obj_RT, wind_total, fuel_total, ratio = solve_RT_for_day(
        d0;
        p_new_star      = p_new_star,
        f_cand_star     = f_cand_star,
        p_wind_cap_star = p_wind_cap_star,
        P_BAT_cap_star  = P_BAT_cap_star,
        E_BAT_cap_star  = E_BAT_cap_star,
        r_cost_up       = r_cost_up,
        r_cost_dn       = r_cost_dn,
        pg_DA           = pg_DA,
    )

    RT_obj[i] = obj_RT
    wind_totals[i] = wind_total
    fuel_totals[i] = fuel_total
    ratios[i] = ratio
end

total_RT = sum(RT_obj)
total_wind_total = sum(wind_totals)
total_fuel_totals = sum(fuel_totals)
ratio_total = total_wind_total / total_fuel_totals

println("Total DA cost over sampled days = ", total_DA)
println("Total RT cost over sampled days = ", total_RT)
println("Total system cost (DA + RT) over sampled days = ", total_DA + total_RT)
println("total_wind_total over sampled days = ", total_wind_total)
println("total_fuel_totals over sampled days = ", total_fuel_totals)
println("ratio_total over sampled days = ", ratio_total)


# --- Cost components (y = 1) ---

gen_cost = sum(p_new_star[i] * c_g_var_id[i] for i in CandGens)

line_cost = sum(f_cand_star[l] * c_l_var_id[l] for l in CandLines)

wind_cost = sum(p_wind_cap_star[g] * cand_wind_data[g].c_var for g in RENW)

bat_energy_cost = sum(E_BAT_cap_star[g] * battery_data[g].c_var_E for g in BATS)

# --- Total  ---
total_cost = gen_cost + line_cost + wind_cost + bat_energy_cost

println("\n=== Investment Cost Breakdown PK (y = 1) ===")
println("Candidate Generators: ", gen_cost)
println("Candidate Lines:      ", line_cost)
println("Wind:                 ", wind_cost)
println("Battery Energy:       ", bat_energy_cost)
println("----------------------------------------")
println("Total:                ", total_cost)
